# TPC Pattern Recognition

In [ ]:
import ROOT as root
import numpy as np
import math
from array import array
from collections import defaultdict
from scipy.spatial import cKDTree
from scipy.optimize import minimize

root.gErrorIgnoreLevel = root.kFatal
%jsroot on


## Input Files

In [ ]:
file_path  = "/Users/mitrankova/Jupyter/PatternRecognition/input/"
file_names = ['Au_Au_seeds_37thevt_66522-0_resid.root']


In [ ]:
open_files = []
trees = {"cluster": [], "residual": [], "hit": [], "vertex": []}

for fname in file_names:
    fpath = file_path + fname
    tfile = root.TFile.Open(fpath, "READ")
    if not tfile or tfile.IsZombie():
        print(f"Failed to open {fpath}")
        for k in trees: trees[k].append(None)
        continue
    open_files.append(tfile)
    trees["cluster"].append(tfile.Get("clustertree"))
    trees["residual"].append(tfile.Get("residualtree"))
    trees["hit"].append(tfile.Get("hittree"))
    trees["vertex"].append(tfile.Get("vertextree"))

hit_tree      = trees["hit"][0]      if trees["hit"]      else None
cluster_tree  = trees["cluster"][0]  if trees["cluster"]  else None
residual_tree = trees["residual"][0] if trees["residual"] else None
vertex_tree   = trees["vertex"][0]   if trees["vertex"]   else None

print(f"Loaded {len(open_files)} file(s)")
print("hit_tree entries     :", hit_tree.GetEntries()      if hit_tree      else 0)
print("cluster_tree entries :", cluster_tree.GetEntries()  if cluster_tree  else 0)
print("residual_tree entries:", residual_tree.GetEntries() if residual_tree else 0)


## Geometry & ADC Thresholds

In [ ]:
# 3 radial modules, 12 phi sectors, 2 sides; 16 layers per module
Npads = [94, 128, 192]
phi_bin_width = [0.0053073, 0.003959, 0.00265145]

module_radius = [
    [29.854978828112735, 31.869737083177956, 32.43665978627038, 33.00171100689825,
     33.56863172731403,  34.133682357783,    34.70060474122243,  35.26565540941076,
     35.83257683544541,  36.39762877363545,  36.964549975549694, 37.52960055896088,
     38.09652180558749,  38.66157293473739,  39.228495272708216, 39.793545257944906],
    [41.65920253621078,  42.67990048015332,  43.7005755287188,   44.7212729094545,
     45.7419615067264,   46.76264656230158,  47.78333428983602,  48.80401878201343,
     49.82471910526506,  50.8454060012135,   51.866093793785126, 52.88677964073831,
     53.90746625152035,  54.92815969895385,  55.948864895868056, 56.9695394315422],
    [58.910963349324035, 60.00800996331871,  61.10505851260341,  62.202104676954924,
     63.29915863086735,  64.39619682986867,  65.49324606923312,  66.59029899562653,
     67.68734047670296,  68.78439383353172,  69.88143340055497,  70.97848786511186,
     72.07553264226554,  73.17257662017182,  74.2696338511705,   75.36667517343196],
]
module_centers   = [34.824262043, 49.3143709839, 67.1388192614]
module_mid_radii = [0.5*(module_centers[i] + module_centers[i+1])
                    for i in range(len(module_centers) - 1)]
TPC_centr = 52.6108270008

ADC_threshold_up   = [20,  100, 1000000, 100000]
ADC_threshold_down = [0,    20,      60,    200]
Select_ADC         = 2   # index into the above arrays


## Coordinate Helpers

In [ ]:
def layer_to_radius(imod, layer_abs):
    iloc = int(layer_abs) - (16*int(imod) + 7)
    if iloc < 0 or iloc >= len(module_radius[int(imod)]):
        return None
    return float(module_radius[int(imod)][iloc])

def radius_to_layer(imod, r):
    rs = np.asarray(module_radius[int(imod)], dtype=float)
    i  = int(np.argmin(np.abs(rs - float(r))))
    return 16*int(imod) + 7 + i, float(rs[i])


## Read Hits

In [ ]:
# key = (sector, imod, side)
# pts_arr[key]         – (N,3) float32:  (tbin, pad, layer)   hardware coords
# adc_arr[key]         – (N,)  int32:     ADC values
# local_coords_arr[key]– (N,3) float64:  (tbin, phi, radius)  local coords
# kdtree[key]          – cKDTree over pts_arr[key]

_points      = defaultdict(list)
_adc         = defaultdict(list)
_local       = defaultdict(list)

if hit_tree:
    for entry in range(int(hit_tree.GetEntries())):
        hit_tree.GetEntry(entry)

        layer = int(hit_tree.layer)
        imod  = (layer - 7) // 16
        if imod < 0 or imod > 2:
            continue

        side   = int(hit_tree.side)
        if side not in (0, 1):
            continue

        adc = int(hit_tree.adc)
        if adc < ADC_threshold_down[Select_ADC]:
            continue

        tbin   = int(hit_tree.tbin)
        if tbin < 0 or tbin > 300:
            continue

        pad    = int(hit_tree.pad)
        sector = int(hit_tree.sector)
        key    = (sector, imod, side)

        _points[key].append((tbin, pad, layer))
        _adc[key].append(adc)
        _local[key].append((tbin,
                            pad * phi_bin_width[imod],
                            module_radius[imod][layer - 7 - imod * 16]))

pts_arr          = {k: np.asarray(v, dtype=np.float32)  for k, v in _points.items()}
adc_arr          = {k: np.asarray(v, dtype=np.int32)    for k, v in _adc.items()}
local_coords_arr = {k: np.asarray(v, dtype=np.float64)  for k, v in _local.items()}
kdtree           = {k: cKDTree(pts_arr[k])              for k in pts_arr}

print(f"Built KD-trees for {len(kdtree)} (sector, imod, side) keys")


In [ ]:
import math
import numpy as np


def wrap_pi(x):
    return (x + math.pi) % (2.0 * math.pi) - math.pi


## Horizontal Chains
Hits in the same layer with |Δpad| ≤ dp and |Δtbin| ≤ dt.

In [ ]:
def _order_chain(key, gis):
    """Sort hit indices by (pad, tbin) for stable drawing."""
    arr   = pts_arr[key]
    order = np.lexsort((arr[gis, 0], arr[gis, 1]))
    return [int(gis[i]) for i in order]


def _build_horizontal_chain(key, seed_gi, used_global, dt=2, dp=1):
    """BFS growth of one horizontal cluster from seed_gi on its layer."""
    seed_gi = int(seed_gi)
    if seed_gi in used_global:
        return [], []

    arr  = pts_arr[key]
    tree = kdtree[key]
    t0, p0, l0 = arr[seed_gi]
    l0i = int(l0)
    r   = float(np.sqrt(dt*dt + dp*dp))

    cluster, consumed = [], []
    queue = [seed_gi]
    used_global.add(seed_gi)
    consumed.append(seed_gi)

    while queue:
        gi = int(queue.pop())
        cluster.append(gi)
        t, p, l = arr[gi]
        for gj in tree.query_ball_point([t, p, l], r=r):
            gj = int(gj)
            if gj in used_global:
                continue
            tj, pj, lj = arr[gj]
            if int(lj) != l0i:
                continue
            if abs(tj - t) <= dt and abs(pj - p) <= dp:
                used_global.add(gj)
                consumed.append(gj)
                queue.append(gj)

    return cluster, consumed


def find_horizontal_chains(key, used_global,
                           dt=2, dp=1,
                           min_hits=3, min_pad_span=5,
                           chain_id_start=0):
    """
    Find all horizontal chains for one key.
    Seeds are all hits sorted by ADC descending.

    Returns (chains, hit_to_chain, next_chain_id).
    Each chain dict contains: chain_id, key, nhits, pad_span, t_span,
    layer_min, layer_max, max_adc, hits (list of hit dicts).
    """
    arr  = pts_arr[key]
    adc  = adc_arr[key]

    order   = np.argsort(adc)[::-1]
    hit_gis = order.astype(np.int32)

    chains, hit_to_chain = [], {}
    chain_id = int(chain_id_start)

    for seed_gi in hit_gis:
        seed_gi = int(seed_gi)
        if seed_gi in used_global:
            continue

        cluster, consumed = _build_horizontal_chain(key, seed_gi, used_global, dt=dt, dp=dp)
        if not cluster:
            continue

        pads     = arr[cluster, 1]
        pad_span = float(np.max(pads) - np.min(pads))

        if len(cluster) < min_hits or pad_span <= min_pad_span:
            for gi in consumed:
                used_global.discard(int(gi))
            continue

        cluster = _order_chain(key, cluster)

        tbins  = arr[cluster, 0]
        layers = arr[cluster, 2]
        hits   = [{"gi": int(gi),
                   "tbin":  int(arr[gi, 0]),
                   "pad":   int(arr[gi, 1]),
                   "layer": int(arr[gi, 2]),
                   "adc":   int(adc[gi])} for gi in cluster]
        for gi in cluster:
            hit_to_chain[int(gi)] = chain_id

        chains.append({
            "chain_id":  chain_id,
            "key":       key,
            "nhits":     len(cluster),
            "pad_span":  int(np.max(pads)   - np.min(pads)),
            "t_span":    int(np.max(tbins)  - np.min(tbins)),
            "layer_min": int(np.min(layers)),
            "layer_max": int(np.max(layers)),
            "max_adc":   int(np.max(adc[cluster])),
            "hits":      hits,
        })
        chain_id += 1

    return chains, hit_to_chain, chain_id


## Sagitta Fit (MLE)
Circle fit in (r, φ) and linear fit in (r, z), both ADC-weighted.

In [ ]:
def _pratt_circle_seed(x, y, w=None):
    """Pratt algebraic circle fit — returns (xc, yc, R) as initial guess."""
    if w is None:
        w = np.ones(len(x))
    w = w / w.sum()
    # moments
    x, y = np.asarray(x, float), np.asarray(y, float)
    Mz  = np.dot(w, x**2 + y**2)
    Mxy = np.dot(w, x*y)
    Mxx = np.dot(w, x**2)
    Myy = np.dot(w, y**2)
    Mx  = np.dot(w, x)
    My  = np.dot(w, y)
    Mzz = np.dot(w, (x**2+y**2)**2)
    Mxz = np.dot(w, x*(x**2+y**2))
    Myz = np.dot(w, y*(x**2+y**2))
    M = np.array([[Mzz, Mxz, Myz, Mz],
                  [Mxz, Mxx, Mxy, Mx],
                  [Myz, Mxy, Myy, My],
                  [Mz,  Mx,  My,  1.0]])
    B = np.array([[0,0,0,-2],[0,1,0,0],[0,0,1,0],[-2,0,0,0]], float)
    try:
        vals, vecs = np.linalg.eig(np.linalg.solve(B, M))
        vals = vals.real
        # pick eigenvector with smallest positive eigenvalue
        pos = vals > 1e-10
        if not pos.any():
            return None
        idx = np.argmin(np.where(pos, vals, np.inf))
        v = vecs[:, idx].real
        xc = -v[1]/(2*v[0]);  yc = -v[2]/(2*v[0])
        R  = np.sqrt(v[1]**2 + v[2]**2 - 4*v[0]*v[3]) / abs(2*v[0])
        return xc, yc, R
    except Exception:
        return None

def smart_adc_weights(adcs, cap_percentile=85, power=0.5, floor_frac=0.15):
    """
    Convert raw ADC to gentler fit weights.

    Steps:
      1. clip to positive
      2. cap very large ADC outliers at a percentile
      3. compress dynamic range with a power < 1
      4. give every hit a nonzero baseline weight

    Returns weights on a relative scale.
    """
    adcs = np.asarray(adcs, dtype=float)
    adcs = np.clip(adcs, 1e-12, None)

    cap = np.percentile(adcs, cap_percentile)
    adc_capped = np.minimum(adcs, cap)

    # compress dynamic range: sqrt by default
    w = adc_capped ** power

    # keep low-charge hits relevant
    wmax = np.max(w) if len(w) else 1.0
    w = np.maximum(w, floor_frac * wmax)

    return w

# ── MLE building blocks ──────────────────────────────────────────────────────

def _mle_nll(d_min, sigma, weights):
    """ADC-weighted NLL: flat inside pad (|d|≤σ), Gaussian outside."""
    u = (d_min / sigma)**2
    return float(np.dot(weights, 0.5 * np.maximum(0.0, u - 1.0)))

def _mle_lgeom(d_min, sigma, weights):
    """Geometric quality variable ∈ (0,1]; 1 = all pads intersected."""
    ln_p = -0.5 * np.maximum(0.0, (d_min/sigma)**2 - 1.0)
    return float(np.exp(np.dot(weights, ln_p) / np.sum(weights)))

# ── Sagitta model ────────────────────────────────────────────────────────────

def _rotate_to_x(x, y):
    """Rotate (x,y) so the best-fit line becomes horizontal."""
    m, b = np.linalg.lstsq(np.vstack([x, np.ones_like(x)]).T, y, rcond=None)[0]
    theta = np.arctan(m)
    c, s  = np.cos(theta), np.sin(theta)
    pts   = np.vstack([x, y - b])
    xr, yr = np.array([[c, s],[-s, c]]) @ pts
    return xr, yr, theta, b

def _sagitta(params, x):
    S, x0, invR = params
    dx = x - x0
    return S - 0.5*invR*dx**2 - (1/8)*invR**3*dx**4 - (1/16)*invR**5*dx**6

# ── Circle fit ───────────────────────────────────────────────────────────────


def fit_sagitta_circle(x, y, sigma_xy=None):
    """
    MLE sagitta circle fit.
    sigma_xy: pad half-widths (1/sqrt(adc) gives ADC weighting).
    Returns (xc, yc, R, sagitta_params, aux, nll, chi2, ndof, lgeom).
    """
    xr, yr, theta, b = _rotate_to_x(x, y)
    if sigma_xy is not None:
        sigma   = np.clip(np.asarray(sigma_xy, float), 1e-12, None)
        weights = 1.0 / sigma**2
    else:
        sigma   = np.ones(len(xr))
        weights = np.ones(len(xr))
    weights = weights / weights.sum() * len(xr)

    seed = _pratt_circle_seed(xr, yr, weights if sigma_xy is not None else None)
    if seed is not None:
        xc_s, yc_s, R_s = seed
        S0   = yc_s + R_s          # sagitta apex
        x0_0 = xc_s                # center x
        invR0 = 1.0 / R_s if R_s > 1 else 0.01
    else:
        S0    = float(np.max(yr))
        x0_0  = float(xr[np.argmax(yr)])
        invR0 = 1/100.0

    def nll(p): return _mle_nll(np.abs(_sagitta(p, xr) - yr), sigma, weights)

    res = minimize(nll,
                  [S0, x0_0, invR0],
                   method="Nelder-Mead",
                   options={"xatol":1e-6,"fatol":1e-6,"maxiter":50000,"maxfev":100000})
    
    #res = minimize(nll,
    #               [float(np.max(yr)), float(xr[np.argmax(yr)]), 1/100.0],
    #               method="Nelder-Mead",
    #               options={"xatol":1e-6,"fatol":1e-6,"maxiter":50000,"maxfev":100000})

    S, x0, invR = res.x
    d_fit  = np.abs(_sagitta(res.x, xr) - yr)
    chi2   = float(np.sum(((d_fit/sigma)**2)))
    ndof   = max(len(xr) - 3, 1)
    R      = 1.0 / invR
    Xc, Yc = x0, S - R
    xc = Xc*math.cos(theta) - Yc*math.sin(theta)
    yc = Xc*math.sin(theta) + Yc*math.cos(theta) + b
    return (xc, yc, R, (S, x0, invR), (xr, yr, theta, b),
            _mle_nll(d_fit, sigma, weights), chi2, ndof,
            _mle_lgeom(d_fit, sigma, weights))

def fit_sagitta_circle_robust(x, y, sigma_xy=None, n_iter=3, sigma_cut=4.0):
    """
    Iterative reweighting: after each fit, down-weight hits with 
    residual > sigma_cut * sigma.  Runs n_iter rounds.
    """
    mask = np.ones(len(x), dtype=bool)
    for iteration in range(n_iter):
        xm, ym = x[mask], y[mask]
        sig_m  = sigma_xy[mask] if sigma_xy is not None else None
        result = fit_sagitta_circle(xm, ym, sigma_xy=sig_m)
        xc, yc, R, sag_p, aux, nll_v, chi2, ndof, lgeom = result
        xr, yr, theta, b = aux

        # compute per-hit residuals in rotated frame
        d_all = np.abs(_sagitta(sag_p, _rotate_to_x(x, y)[0]) - _rotate_to_x(x, y)[1])
        sigma_all = sigma_xy if sigma_xy is not None else np.ones(len(x))
        sigma_all = np.clip(sigma_all, 1e-12, None)

        mask = (d_all / sigma_all) < sigma_cut
        if mask.sum() < 3:
            mask = np.ones(len(x), dtype=bool)  # revert if too few hits left
            break

    # final fit on surviving hits
    xm, ym   = x[mask], y[mask]
    sig_m    = sigma_xy[mask] if sigma_xy is not None else None
    return fit_sagitta_circle(xm, ym, sigma_xy=sig_m)

# ── Perigee conversion ────────────────────────────────────────────────────────

def circle_to_perigee(xc, yc, R, invR):
    """Circle (xc,yc,R) → (dca, d0, phi0, kappa, xp, yp)."""
    D     = math.hypot(xc, yc)
    Rabs  = abs(R)
    sign_k = math.copysign(1.0, invR) if invR != 0 else 1.0
    if D < 1e-12 or Rabs < 1e-12:
        xp, yp = Rabs, 0.0
    else:
        ux, uy = xc/D, yc/D
        xp = xc - ux*Rabs if D >= Rabs else xc + ux*Rabs
        yp = yc - uy*Rabs if D >= Rabs else yc + uy*Rabs
    dca = math.hypot(xp, yp)
    d0  = sign_k * dca
    rx, ry = xp - xc, yp - yc
    rn = math.hypot(rx, ry)
    if rn < 1e-12:
        tx, ty = 0.0, 1.0
    elif sign_k > 0:
        tx, ty = -ry/rn,  rx/rn
    else:
        tx, ty =  ry/rn, -rx/rn
    return dca, d0, math.atan2(ty, tx), invR, xp, yp

def fit_sagitta_perigee(x, y, sigma_xy=None):
    #xc, yc, R, sag_p, aux, nll, chi2, ndof, lgeom = fit_sagitta_circle(
    #    x, y, sigma_xy=sigma_xy)
    xc, yc, R, sag_p, aux, nll, chi2, ndof, lgeom = fit_sagitta_circle_robust(
        x, y, sigma_xy=sigma_xy)
    dca, d0, phi0, kappa, xp, yp = circle_to_perigee(xc, yc, R, sag_p[2])
    return dca, d0, phi0, kappa, xp, yp, (xc, yc, R), (sag_p, aux), nll, chi2, ndof, lgeom

# ── Z-r linear fit ────────────────────────────────────────────────────────────

def fit_zr_linear(x, y, z, weights=None):
    """MLE linear fit z(r) = m*r + c. Returns (m, c, nll, chi2, ndof, r, z_fit, lgeom)."""
    x, y, z = np.asarray(x,float), np.asarray(y,float), np.asarray(z,float)
    r = np.sqrt(x**2 + y**2)
    if weights is not None:
        w      = np.clip(np.asarray(weights,float), 1e-12, None)
        sigma  = 1.0 / np.sqrt(w)
        mle_w  = w / w.sum() * len(r)
    else:
        sigma  = np.ones(len(r))
        mle_w  = np.ones(len(r))
    m0, c0 = np.linalg.lstsq(np.vstack([r,np.ones_like(r)]).T, z, rcond=None)[0]
    def nll(p): return _mle_nll(np.abs(z-(p[0]*r+p[1])), sigma, mle_w)
    res = minimize(nll, [m0, c0], method="Nelder-Mead",
                   options={"xatol":1e-6,"fatol":1e-6,"maxiter":20000,"maxfev":40000})
    m, c   = res.x
    z_fit  = m*r + c
    d_fin  = np.abs(z - z_fit)
    return (m, c,
            _mle_nll(d_fin, sigma, mle_w),
            float(np.sum(d_fin**2)),
            max(len(z)-2,1),
            r, z_fit,
            _mle_lgeom(d_fin, sigma, mle_w))

# ── Geometry helpers ──────────────────────────────────────────────────────────

def wrap_phi(phi):
    while phi >=  math.pi: phi -= 2*math.pi
    while phi < -math.pi:  phi += 2*math.pi
    return phi

def circle_phi_at_radius(xc, yc, R, r_hit, phi_ref):
    """Return the intersection phi of circle (xc,yc,R) with cylinder r_hit
    closest to phi_ref, or None if no real intersection."""
    Rabs = abs(R); D = math.hypot(xc, yc)
    if D < 1e-12 or Rabs < 1e-12: return None
    if D > Rabs+r_hit or D < abs(Rabs-r_hit): return None
    ex, ey = xc/D, yc/D
    a  = (r_hit**2 - Rabs**2 + D**2) / (2*D)
    h2 = r_hit**2 - a**2
    if h2 < 0:
        if h2 > -1e-10: h2 = 0.0
        else: return None
    h  = math.sqrt(h2)
    px, py = -ey, ex
    phiA = math.atan2(a*ey + h*py, a*ex + h*px)
    phiB = math.atan2(a*ey - h*py, a*ex - h*px)
    return phiA if abs(wrap_phi(phi_ref-phiA)) <= abs(wrap_phi(phi_ref-phiB)) else phiB

def zr_to_z0_theta(m_zr, c_zr, xp, yp, mod):
    r_p = math.hypot(xp, yp)
    z0  = m_zr*r_p + c_zr
    z_centr     = m_zr*module_centers[mod] + c_zr
    z_tpc_centr = m_zr*TPC_centr + c_zr
    z_zero      = c_zr
    lam = math.atan(m_zr)
    return z0, math.pi/2 - lam, z_zero, z_centr, z_tpc_centr, lam

def weighted_linear_fit(x, y, w=None):
    """
    Weighted fit: y = m*x + c
    Returns (m, c, chi2, ndof)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2:
        return None

    if w is None:
        w = np.ones_like(x, dtype=float)
    else:
        w = np.asarray(w, dtype=float)
        w = np.clip(w, 1e-12, None)

    S   = np.sum(w)
    Sx  = np.sum(w * x)
    Sy  = np.sum(w * y)
    Sxx = np.sum(w * x * x)
    Sxy = np.sum(w * x * y)

    den = S * Sxx - Sx * Sx
    if abs(den) < 1e-12:
        return None

    m = (S * Sxy - Sx * Sy) / den
    c = (Sy - m * Sx) / S

    yhat = m * x + c
    chi2 = float(np.sum(w * (y - yhat) ** 2))
    ndof = max(len(x) - 2, 1)

    return m, c, chi2, ndof
# ── Main chain fitter ─────────────────────────────────────────────────────────
def fit_chain_with_sagitta(chain, local_coords_arr, adc_arr):
    key = chain["key"]

    x, y, z, adcs = [], [], [], []
    r_hits, phi_local_hits = [], []

    for gi in chain["chain_consumed"]:
        t, phi_local, r = local_coords_arr[key][gi]

        x.append(float(r * np.cos(phi_local)))
        y.append(float(r * np.sin(phi_local)))
        z.append(float(t))
        adcs.append(float(adc_arr[key][gi]))

        r_hits.append(float(r))
        phi_local_hits.append(float(phi_local))

    if len(x) < 3:
        return None

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    z = np.asarray(z, dtype=float)
    r_hits = np.asarray(r_hits, dtype=float)
    phi_local_hits = np.asarray(phi_local_hits, dtype=float)
    adcs = np.clip(np.asarray(adcs, dtype=float), 1e-12, None)

    # transverse sagitta fit
    #(dca, d0, phi0, kappa, xp, yp,
    # (xc, yc, R), _, nll_sag, chi2_sag, ndof_sag, lgeom_sag
    #) = fit_sagitta_perigee(x, y, sigma_xy=1.0 / np.sqrt(adcs))
    fit_w = smart_adc_weights(adcs, cap_percentile=85, power=0.5, floor_frac=0.15)

    # transverse sagitta fit
    (dca, d0, phi0, kappa, xp, yp,
    (xc, yc, R), _, nll_sag, chi2_sag, ndof_sag, lgeom_sag
    ) = fit_sagitta_perigee(x, y, sigma_xy=1.0 / np.sqrt(fit_w))

    # tbin(r)
    #zr_fit = weighted_linear_fit(r_hits, z, adcs)
    zr_fit   = weighted_linear_fit(r_hits, z, fit_w)
    if zr_fit is None:
        return None
    m_zr, c_zr, chi2_zr, ndof_zr = zr_fit

    # phi_local(r)
    #phir_fit = weighted_linear_fit(r_hits, phi_local_hits, adcs)
    phir_fit = weighted_linear_fit(r_hits, phi_local_hits, fit_w)
    if phir_fit is None:
        return None
    m_phir, c_phir, chi2_phir, ndof_phir = phir_fit

    # keep your existing theta/z0 logic if needed
    z0, theta_old, z_zero, z_centr, z_tpc_centr, lam = zr_to_z0_theta(
        m_zr, c_zr, xp, yp, key[1]
    )


    return {
        "dca": float(dca),
        "d0": float(d0),
        "z0": float(z0),
        "phi0": float(phi0),
        "theta": float(math.atan(m_zr)),   # matching theta
        "kappa": float(kappa),
        "z_zero": float(z_zero),
        "z_centr": float(z_centr),
        "z_tpc_centr": float(z_tpc_centr),
        "xp": float(xp),
        "yp": float(yp),
        "xc": float(xc),
        "yc": float(yc),
        "R": float(R),

        "chi2_sag": float(chi2_sag),
        "ndof_sag": int(ndof_sag),

        "m_zr": float(m_zr),
        "c_zr": float(c_zr),
        "chi2_zr": float(chi2_zr),
        "ndof_zr": int(ndof_zr),

        "m_phir": float(m_phir),
        "c_phir": float(c_phir),
        "chi2_phir": float(chi2_phir),
        "ndof_phir": int(ndof_phir),

        "alpha": float(math.atan(m_phir)),  # matching alpha
        "lambda": float(lam),
        "fit_success": True,
    }

def refit_collection(hit_collections, local_coords_arr, adc_arr):
    """
    Refit every track in hit_collections and store the result in track['refit'].

    Stored fit contains:
      * original sagitta/circle fit (kept for drawing if needed)
      * tbin(r) linear fit
      * local phi(r) linear fit
      * tangent-angle parameters for matching
    """
    ok = 0

    for track in hit_collections:
        x, y = [], []
        tbin_hits, phi_local_hits, r_hits, adcs = [], [], [], []

        for ch in track["chains"]:
            key = ch["key"]
            for gi in ch["chain_consumed"]:
                t, phi_local, r = local_coords_arr[key][gi]

                r = float(r)
                t = float(t)
                phi_local = float(phi_local)
                adc = float(adc_arr[key][gi])

                x.append(r * np.cos(phi_local))
                y.append(r * np.sin(phi_local))

                tbin_hits.append(t)
                phi_local_hits.append(phi_local)
                r_hits.append(r)
                adcs.append(adc)

        if len(r_hits) < 3:
            track["refit"] = None
            continue

        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        r_hits = np.asarray(r_hits, dtype=float)
        tbin_hits = np.asarray(tbin_hits, dtype=float)
        phi_local_hits = np.asarray(phi_local_hits, dtype=float)
        adcs = np.clip(np.asarray(adcs, dtype=float), 1e-12, None)

        try:
            # keep transverse sagitta fit for optional drawing / diagnostics
            #(dca, d0, phi0, kappa, xp, yp,
            # (xc, yc, R), _, nll_sag, chi2_sag, ndof_sag, lgeom_sag
            #) = fit_sagitta_perigee(x, y, sigma_xy=1.0 / np.sqrt(adcs))
            fit_w = smart_adc_weights(adcs, cap_percentile=85, power=0.5, floor_frac=0.15)
            (dca, d0, phi0, kappa, xp, yp,
            (xc, yc, R), _, nll_sag, chi2_sag, ndof_sag, lgeom_sag
            ) = fit_sagitta_perigee(x, y, sigma_xy=1.0 / np.sqrt(fit_w))

            #zr_fit = weighted_linear_fit(r_hits, tbin_hits, adcs)
            #phir_fit = weighted_linear_fit(r_hits, phi_local_hits, adcs)
            zr_fit = weighted_linear_fit(r_hits, tbin_hits, fit_w)
            phir_fit = weighted_linear_fit(r_hits, phi_local_hits, fit_w)

            if zr_fit is None or phir_fit is None:
                track["refit"] = None
                continue

            m_zr, c_zr, chi2_zr, ndof_zr = zr_fit
            m_phir, c_phir, chi2_phir, ndof_phir = phir_fit

            # matching angles in the two projected views
            theta_proj = math.atan(m_zr)
            alpha_proj = math.atan(m_phir)

            track["refit"] = {
                # keep original sagitta information
                "dca": float(dca),
                "d0": float(d0),
                "phi0": float(phi0),
                "kappa": float(kappa),
                "xp": float(xp),
                "yp": float(yp),
                "xc": float(xc),
                "yc": float(yc),
                "R": float(R),
                "nll_sag": float(nll_sag),
                "chi2_sag": float(chi2_sag),
                "ndof_sag": int(ndof_sag),
                "lgeom_sag": float(lgeom_sag),

                # projected fits actually used for matching
                "m_zr": float(m_zr),
                "c_zr": float(c_zr),
                "chi2_zr": float(chi2_zr),
                "ndof_zr": int(ndof_zr),

                "m_phir": float(m_phir),
                "c_phir": float(c_phir),
                "chi2_phir": float(chi2_phir),
                "ndof_phir": int(ndof_phir),

                # these are the matching-angle definitions
                "theta": float(theta_proj),
                "alpha": float(alpha_proj),

                # useful debug / branch sanity
                "phi_local_min": float(np.min(phi_local_hits)),
                "phi_local_max": float(np.max(phi_local_hits)),
                "phi_local_median": float(np.median(phi_local_hits)),
                "r_min": float(np.min(r_hits)),
                "r_max": float(np.max(r_hits)),
            }
            ok += 1

        except Exception:
            track["refit"] = None

    print(f"Refit successful for {ok}/{len(hit_collections)} tracks")

def refit_collection_with_extra_hits(hit_collections, local_coords_arr, adc_arr):
    """
    Refit every track and include:
      - all chain_consumed hits
      - all extra hits from track["extra_hits_by_key"]

    Stores result in track["refit"].
    """
    ok = 0

    for track in hit_collections:
        x, y = [], []
        tbin_hits, phi_local_hits, r_hits, adcs = [], [], [], []

        seen = set()

        # regular chain hits
        for ch in track["chains"]:
            key = ch["key"]
            for gi in ch["chain_consumed"]:
                gi = int(gi)
                tag = (key, gi)
                if tag in seen:
                    continue
                seen.add(tag)

                t, phi_local, r = local_coords_arr[key][gi]
                adc = float(adc_arr[key][gi])

                r = float(r)
                t = float(t)
                phi_local = float(phi_local)

                x.append(r * np.cos(phi_local))
                y.append(r * np.sin(phi_local))
                tbin_hits.append(t)
                phi_local_hits.append(phi_local)
                r_hits.append(r)
                adcs.append(adc)

        # extra tube hits
        for key, gi_list in track.get("extra_hits_by_key", {}).items():
            for gi in gi_list:
                gi = int(gi)
                tag = (key, gi)
                if tag in seen:
                    continue
                seen.add(tag)

                t, phi_local, r = local_coords_arr[key][gi]
                adc = float(adc_arr[key][gi])

                r = float(r)
                t = float(t)
                phi_local = float(phi_local)

                x.append(r * np.cos(phi_local))
                y.append(r * np.sin(phi_local))
                tbin_hits.append(t)
                phi_local_hits.append(phi_local)
                r_hits.append(r)
                adcs.append(adc)

        if len(r_hits) < 3:
            track["refit"] = None
            continue

        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        r_hits = np.asarray(r_hits, dtype=float)
        tbin_hits = np.asarray(tbin_hits, dtype=float)
        phi_local_hits = np.asarray(phi_local_hits, dtype=float)
        adcs = np.clip(np.asarray(adcs, dtype=float), 1e-12, None)

        try:
            fit_w = smart_adc_weights(adcs, cap_percentile=85, power=0.5, floor_frac=0.15)

            (dca, d0, phi0, kappa, xp, yp,
             (xc, yc, R), _, nll_sag, chi2_sag, ndof_sag, lgeom_sag
            ) = fit_sagitta_perigee(x, y, sigma_xy=1.0 / np.sqrt(fit_w))

            zr_fit   = weighted_linear_fit(r_hits, tbin_hits, fit_w)
            phir_fit = weighted_linear_fit(r_hits, phi_local_hits, fit_w)

            if zr_fit is None or phir_fit is None:
                track["refit"] = None
                continue

            m_zr, c_zr, chi2_zr, ndof_zr = zr_fit
            m_phir, c_phir, chi2_phir, ndof_phir = phir_fit

            theta_proj = math.atan(m_zr)
            alpha_proj = math.atan(m_phir)

            track["refit"] = {
                "dca": float(dca),
                "d0": float(d0),
                "phi0": float(phi0),
                "kappa": float(kappa),
                "xp": float(xp),
                "yp": float(yp),
                "xc": float(xc),
                "yc": float(yc),
                "R": float(R),
                "nll_sag": float(nll_sag),
                "chi2_sag": float(chi2_sag),
                "ndof_sag": int(ndof_sag),
                "lgeom_sag": float(lgeom_sag),

                "m_zr": float(m_zr),
                "c_zr": float(c_zr),
                "chi2_zr": float(chi2_zr),
                "ndof_zr": int(ndof_zr),

                "m_phir": float(m_phir),
                "c_phir": float(c_phir),
                "chi2_phir": float(chi2_phir),
                "ndof_phir": int(ndof_phir),

                "theta": float(theta_proj),
                "alpha": float(alpha_proj),

                "phi_local_min": float(np.min(phi_local_hits)),
                "phi_local_max": float(np.max(phi_local_hits)),
                "phi_local_median": float(np.median(phi_local_hits)),
                "r_min": float(np.min(r_hits)),
                "r_max": float(np.max(r_hits)),
            }
            ok += 1

        except Exception:
            track["refit"] = None

    print(f"Refit successful for {ok}/{len(hit_collections)} tracks")

## Vertical Chains
Cross-layer chains grown from local ADC maxima.

In [ ]:
# ── helpers ──────────────────────────────────────────────────────────────────

def _wls_line(x, y, w):
    """Weighted least-squares y = a + b*x. Returns (a, b)."""
    sw = sx = sy = sxx = sxy = 0.0
    for xi, yi, wi in zip(x, y, w):
        wi = float(wi)
        if wi <= 0: continue
        sw += wi; sx += wi*xi; sy += wi*yi
        sxx += wi*xi*xi; sxy += wi*xi*yi
    if sw <= 0: return 0.0, 0.0
    denom = sw*sxx - sx*sx
    if denom == 0: return sy/sw, 0.0
    b = (sw*sxy - sx*sy) / denom
    return (sy - b*sx) / sw, b


def _fit_from_pts(pts):
    """pts = [(L, t, p, w), ...] → (at, bt, ap, bp) weighted line fits."""
    xs = [float(L) for L,t,p,w in pts]
    ts = [float(t) for L,t,p,w in pts]
    ps = [float(p) for L,t,p,w in pts]
    ws = [float(w) for L,t,p,w in pts]
    return _wls_line(xs, ts, ws) + _wls_line(xs, ps, ws)


def is_local_max(key, j):
    """True if hit j is a strict 3×3 local ADC maximum on its layer."""
    arr  = pts_arr[key]
    adc  = adc_arr[key]
    tree = kdtree[key]
    t0, p0, l0 = arr[j]
    a0 = int(adc[j])
    for k in tree.query_ball_point([t0, p0, l0], r=math.sqrt(2.0)):
        if k == j: continue
        t, p, l = arr[k]
        if int(l) != int(l0): continue
        if abs(t-t0) <= 1 and abs(p-p0) <= 1:
            if int(adc[k]) >= a0:
                return False
    return True


def find_hit_group(key, p0, t0, l0, dp, dt, exclude_gi):
    """
    Return (closest_hit, local_max_hit, support_list) within a box on layer l0.
    All dicts contain keys: gi, t, p, l, adc.  closest_hit also has dist2.
    """
    arr  = pts_arr[key]
    adcs = adc_arr[key]
    t = arr[:, 0].astype(float)
    p = arr[:, 1].astype(float)
    l = arr[:, 2].astype(float)
    mask = (np.abs(p - p0) <= dp) & (np.abs(t - t0) <= dt) & (l == l0)
    idx  = np.flatnonzero(mask)

    support = []
    for gi in idx:
        gi = int(gi)
        if gi in exclude_gi: continue
        support.append({"gi": gi, "t": float(t[gi]), "p": float(p[gi]),
                        "l": float(l[gi]), "adc": int(adcs[gi])})
    if not support:
        return None, None, []

    P  = np.array([h["p"] for h in support])
    T  = np.array([h["t"] for h in support])
    L  = np.array([h["l"] for h in support])
    d2 = (P-p0)**2 + (T-t0)**2 + (L-l0)**2

    i_close = int(np.argmin(d2))
    closest = {**support[i_close], "dist2": float(d2[i_close])}
    i_max   = int(np.argmax([h["adc"] for h in support]))
    local_max = support[i_max].copy()

    for h, dd in zip(support, d2):
        h["dist2"] = float(dd)
    support.sort(key=lambda h: (h["dist2"], -h["adc"]))
    return closest, local_max, support



In [ ]:
def _add_blob_to_chain(key, main_gi, sup_gis,
                       chain_main, chain_consumed, used_global):
    """
    Add a blob (main hit + satellite hits with decreasing ADC) to a chain.
    Grows outward from main_gi while ADC is non-increasing.
    """
    main_gi = int(main_gi)
    if main_gi in used_global: return

    sup_gis = [int(s) for s in sup_gis]
    if any(s in used_global for s in sup_gis): return

    used_global.add(main_gi)
    chain_main.append(main_gi)
    chain_consumed.append(main_gi)
    for s in sup_gis:
        used_global.add(s)
        chain_consumed.append(s)

    # grow with ADC falloff in (t, p) directions on the same layer
    arr     = pts_arr[key]
    adc     = adc_arr[key]
    idx_map = {(int(t), int(p), int(l)): gi
               for gi, (t, p, l) in enumerate(arr)}
    t0, p0, l0 = map(int, arr[main_gi])

    dirs = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    state = {d: [t0, p0, int(adc[main_gi]), True] for d in dirs}

    for _ in range(50):
        if not any(s[3] for s in state.values()): break
        for (dt, dp) in dirs:
            s = state[(dt, dp)]
            if not s[3]: continue
            nt, np_, curr_adc = s[0]+dt, s[1]+dp, s[2]
            gj = idx_map.get((nt, np_, l0))
            if gj is None or gj in used_global:
                s[3] = False; continue
            new_adc = int(adc[gj])
            if new_adc <= curr_adc:
                used_global.add(gj)
                chain_main.append(gj)
                chain_consumed.append(gj)
                s[0], s[1], s[2] = nt, np_, new_adc
            else:
                s[3] = False


def build_vertical_chain(key, seed_hit, seed_support, used_global,
                         dt=2, dp=2, dl=1, max_skip=1):
    """
    Build one vertical chain starting from seed_hit (a hit-dict with 'gi').

    Steps:
      1. Add seed blob (ADC-falloff growth).
      2. Find blob on next layer near the seed.
      3. Fit the pair with ADC-weighted straight lines in (L,t) and (L,p).
      4. Propagate fit layer by layer; accept blobs within (dt, dp) of prediction.

    Returns (chain_main, chain_consumed, fit_history_hw, fit_history_local).
    """
    arr   = pts_arr[key]
    chain_main, chain_consumed = [], []
    fit_hist_hw, fit_hist_loc  = [], []

    main_gi  = int(seed_hit["gi"])
    sup_gis  = [int(h["gi"]) for h in seed_support if h["gi"] != main_gi]
    _add_blob_to_chain(key, main_gi, sup_gis,
                       chain_main, chain_consumed, used_global)

    t0, p0, l0 = arr[main_gi]

    # find first next-layer blob
    next_max = None
    for skip in range(max_skip + 1):
        _, next_max, next_sup = find_hit_group(
            key, p0, t0, l0 + dl*(skip+1), dp=dp, dt=dt, exclude_gi=used_global)
        if next_max is not None:
            l0_actual = l0 + dl*(skip+1)
            break
    if next_max is None:
        return chain_main, chain_consumed, fit_hist_hw, fit_hist_loc

    next_gi   = int(next_max["gi"])
    next_sups = [int(h["gi"]) for h in next_sup if h["gi"] != next_gi]
    _add_blob_to_chain(key, next_gi, next_sups,
                       chain_main, chain_consumed, used_global)

    def _store_fit(coords_arr, fhist):
        pts  = ([(float(coords_arr[key][main_gi][2]),
                  float(coords_arr[key][main_gi][0]),
                  float(coords_arr[key][main_gi][1]),
                  float(adc_arr[key][main_gi]))]
                + [(float(coords_arr[key][s][2]),
                    float(coords_arr[key][s][0]),
                    float(coords_arr[key][s][1]),
                    float(adc_arr[key][s])) for s in sup_gis]
                + [(float(coords_arr[key][next_gi][2]),
                    float(coords_arr[key][next_gi][0]),
                    float(coords_arr[key][next_gi][1]),
                    float(adc_arr[key][next_gi]))]
                + [(float(coords_arr[key][s][2]),
                    float(coords_arr[key][s][0]),
                    float(coords_arr[key][s][1]),
                    float(adc_arr[key][s])) for s in next_sups])
        at, bt, ap, bp = _fit_from_pts(pts)
        fhist.append({"t_fit": {"a": at, "b": bt},
                      "p_fit": {"a": ap, "b": bp}})
        return at, bt, ap, bp

    at, bt, ap, bp = _store_fit(pts_arr, fit_hist_hw)
    _store_fit(local_coords_arr, fit_hist_loc)

    cur_layer   = l0_actual
    cur_main_gi = next_gi
    cur_sup     = next_sups
    misses      = 0

    while True:
        target = cur_layer + dl
        L_min  = 16*key[1] + 7
        L_max  = 16*(key[1]+1) + 7
        if target < L_min or target >= L_max:
            break

        found_max = None
        for skip in range(max_skip + 1):
            sl = target + dl*skip
            if sl < L_min or sl >= L_max: continue
            t_pred = at + bt*float(sl)
            p_pred = ap + bp*float(sl)
            _, found_max, found_sup = find_hit_group(
                key, p_pred, t_pred, sl, dp=dp, dt=dt, exclude_gi=used_global)
            if found_max is not None:
                target = sl; misses = 0; break

        if found_max is None:
            misses += 1
            if misses > max_skip: break
            cur_layer = target
            continue

        f_gi   = int(found_max["gi"])
        f_sups = [int(h["gi"]) for h in found_sup if h["gi"] != f_gi]
        _add_blob_to_chain(key, f_gi, f_sups,
                           chain_main, chain_consumed, used_global)

        # update fit using last two blobs
        pts_hw = ([(float(arr[cur_main_gi][2]),
                    float(arr[cur_main_gi][0]),
                    float(arr[cur_main_gi][1]),
                    float(adc_arr[key][cur_main_gi]))]
                  + [(float(arr[s][2]), float(arr[s][0]), float(arr[s][1]),
                      float(adc_arr[key][s])) for s in cur_sup]
                  + [(float(arr[f_gi][2]), float(arr[f_gi][0]),
                      float(arr[f_gi][1]), float(adc_arr[key][f_gi]))]
                  + [(float(arr[s][2]), float(arr[s][0]), float(arr[s][1]),
                      float(adc_arr[key][s])) for s in f_sups])
        at, bt, ap, bp = _fit_from_pts(pts_hw)
        fit_hist_hw.append({"t_fit": {"a": at, "b": bt},
                            "p_fit": {"a": ap, "b": bp}})

        pts_loc = ([(float(local_coords_arr[key][cur_main_gi][2]),
                     float(local_coords_arr[key][cur_main_gi][0]),
                     float(local_coords_arr[key][cur_main_gi][1]),
                     float(adc_arr[key][cur_main_gi]))]
                   + [(float(local_coords_arr[key][s][2]),
                       float(local_coords_arr[key][s][0]),
                       float(local_coords_arr[key][s][1]),
                       float(adc_arr[key][s])) for s in cur_sup]
                   + [(float(local_coords_arr[key][f_gi][2]),
                       float(local_coords_arr[key][f_gi][0]),
                       float(local_coords_arr[key][f_gi][1]),
                       float(adc_arr[key][f_gi]))]
                   + [(float(local_coords_arr[key][s][2]),
                       float(local_coords_arr[key][s][0]),
                       float(local_coords_arr[key][s][1]),
                       float(adc_arr[key][s])) for s in f_sups])
        fit_hist_loc.append({"t_fit": {"a": _fit_from_pts(pts_loc)[0],
                                        "b": _fit_from_pts(pts_loc)[1]},
                             "p_fit": {"a": _fit_from_pts(pts_loc)[2],
                                        "b": _fit_from_pts(pts_loc)[3]}})

        cur_layer   = target
        cur_main_gi = f_gi
        cur_sup     = f_sups

    return chain_main, chain_consumed, fit_hist_hw, fit_hist_loc


def find_vertical_chains(key, used_global,
                         dt=2, dp=2, min_hits=3,
                         chain_id_start=0):
    """
    Find all vertical chains for one key.
    Seeds at local ADC maxima; chains grown with build_vertical_chain;
    each chain is immediately fitted with fit_chain_with_sagitta.

    Returns list of chain dicts.
    """
    arr  = pts_arr[key]
    adc  = adc_arr[key]
    chains = []
    cid    = int(chain_id_start)

    for j in range(len(arr)):
        if not is_local_max(key, j): continue
        t0, p0, l0 = arr[j]

        _, peak, group = find_hit_group(
            key, p0, t0, l0, dp=3, dt=2, exclude_gi=used_global)
        if peak is None: continue

        cm, cc, fh_hw, fh_loc = build_vertical_chain(
            key, peak, group, used_global, dt=dt, dp=dp, dl=1)
        if len(cm) < min_hits: continue

        chain_rec = {
            "chain_id":       cid,
            "key":            key,
            "chain_main":     cm,
            "chain_consumed": cc,
            "chain_satellites": [],
            "fit_history_hw": fh_hw,
            "fit_history_loc": fh_loc,
        }
        chain_rec["sagitta_fit"] = fit_chain_with_sagitta(
            chain_rec, local_coords_arr, adc_arr=adc_arr)
        chains.append(chain_rec)
        cid += 1

    print(f"Found {len(chains)} vertical chains for key={key}")
    return chains


In [ ]:
def attach_satellites(key, vertical_chains, used_global, dt=6, dp=6):
    """
    For each main hit in each chain, collect unused neighbours as satellites.
    Satellites are added to chain['chain_satellites'] and marked used.
    """
    for ch in vertical_chains:
        for gi in ch["chain_main"]:
            gi  = int(gi)
            t, p, l = pts_arr[key][gi]
            _, _, support = find_hit_group(
                key, p, t, l, dp=dp, dt=dt, exclude_gi=used_global)
            for h in support:
                hgi = int(h["gi"])
                used_global.add(hgi)
                ch["chain_satellites"].append(hgi)


In [ ]:
def grow_full_tracks_with_tube_hits(
    full_tracks,
    sector,
    side,
    t_window=3.0,
    phi_window=0.010,
    r_margin=2.0,
    max_iterations=1,
    debug=True
):
    """
    Grow each full track by collecting extra leftover hits inside a tube around
    the current full-track fit in (tbin, local phi) as functions of radius.

    Rules:
      - hit must be within |t - t_pred(r)| < t_window
      - hit must be within |phi - phi_pred(r)| < phi_window
      - hit must not already belong to another full track
      - hit may already belong to this same full track (ignored)

    Adds accepted hits into:
      track["extra_hits_by_key"][key] = [gi, ...]

    Then refits tracks including those extra hits.
    """

    def _track_owned_hits(track):
        owned = set()
        for ch in track["chains"]:
            key = ch["key"]
            for gi in ch["chain_consumed"]:
                owned.add((key, int(gi)))
        for key, gis in track.get("extra_hits_by_key", {}).items():
            for gi in gis:
                owned.add((key, int(gi)))
        return owned

    # initialize storage
    for trk in full_tracks:
        if "extra_hits_by_key" not in trk:
            trk["extra_hits_by_key"] = {}

    for it in range(max_iterations):
        # ownership map from current full-track content
        owner_map = {}
        for itrk, trk in enumerate(full_tracks):
            for tag in _track_owned_hits(trk):
                owner_map[tag] = itrk

        n_added_total = 0

        for itrk, trk in enumerate(full_tracks):
            fit = trk.get("refit", None)
            if fit is None:
                continue

            # modules touched by this full track
            keys_in_track = sorted(set(ch["key"] for ch in trk["chains"]))

            # current owned hits of this same track
            own_hits = _track_owned_hits(trk)

            rmin = float(fit["r_min"]) - float(r_margin)
            rmax = float(fit["r_max"]) + float(r_margin)

            newly_added = {}

            for key in keys_in_track:
                if key[0] != sector or key[2] != side:
                    continue

                arr = local_coords_arr[key]

                for gi in range(len(arr)):
                    tag = (key, int(gi))

                    # skip hits belonging to another full track
                    if tag in owner_map and owner_map[tag] != itrk:
                        continue

                    # skip already owned by this track
                    if tag in own_hits:
                        continue

                    t, phi_local, r = arr[gi]
                    t = float(t)
                    phi_local = float(phi_local)
                    r = float(r)

                    if r < rmin or r > rmax:
                        continue

                    t_pred   = float(fit["m_zr"] * r + fit["c_zr"])
                    phi_pred = float(fit["m_phir"] * r + fit["c_phir"])

                    dt   = abs(t - t_pred)
                    dphi = abs(wrap_pi(phi_local - phi_pred))

                    if dt <= t_window and dphi <= phi_window:
                        newly_added.setdefault(key, []).append(int(gi))

            # commit newly found hits for this track
            n_added_here = 0
            for key, gis in newly_added.items():
                prev = set(int(x) for x in trk["extra_hits_by_key"].get(key, []))
                prev.update(int(x) for x in gis)
                trk["extra_hits_by_key"][key] = sorted(prev)
                n_added_here += len(gis)

            n_added_total += n_added_here

            if debug:
                print(f"[tube-grow] iter={it+1} track_id={trk['track_id']} added {n_added_here} hits")

        # refit all tracks after growth
        refit_collection_with_extra_hits(full_tracks, local_coords_arr, adc_arr)

        if debug:
            print(f"[tube-grow] iteration {it+1}: total added = {n_added_total}")

        if n_added_total == 0:
            break

    return full_tracks

## Tilted chains

In [ ]:
import copy
import math
import numpy as np


def build_blob_groups_from_horizontal_chains(
    key,
    used_blocked,
    h_dt=2,
    h_dp=2,
    h_min_hits=3,
    h_min_pad_span=5,
    chain_id_start=200000
):
    """
    Build 'blob groups' for pass2 from leftover hits.

    used_blocked:
        hits forbidden for pass2, e.g. hits already used by first-pass full tracks

    Returns:
        blob_groups: list of dicts
        horiz_chains: original horizontal chains found on leftovers

    Each blob_group has:
        {
          "main_gi": int,
          "support_gis": [int, ...],   # includes all hits in the blob except main
          "all_gis": [int, ...],
          "layer": int,
          "t": float,
          "p": float,
          "adc": int,
          "kind": "horizontal" or "single"
        }
    """
    arr = pts_arr[key]
    adc = adc_arr[key]

    # temporary used set only for finding horizontal chains among leftovers
    tmp_used = set(used_blocked)

    horiz_chains, _, _ = find_horizontal_chains(
        key, tmp_used,
        dt=h_dt, dp=h_dp,
        min_hits=h_min_hits,
        min_pad_span=h_min_pad_span,
        chain_id_start=chain_id_start
    )

    blob_groups = []
    hits_in_hchains = set()

    # convert each horizontal chain into one blob
    for hc in horiz_chains:
        gis = [int(h["gi"]) for h in hc["hits"]]
        if not gis:
            continue

        hits_in_hchains.update(gis)

        # choose main hit = max ADC hit inside horizontal blob
        main_gi = max(gis, key=lambda gi: int(adc[gi]))
        support_gis = [gi for gi in gis if gi != main_gi]

        # weighted center is useful for prediction/matching
        w = np.array([float(adc[gi]) for gi in gis], dtype=float)
        tsum = np.sum(w * arr[gis, 0].astype(float))
        psum = np.sum(w * arr[gis, 1].astype(float))
        wsum = np.sum(w) if np.sum(w) > 0 else 1.0

        blob_groups.append({
            "main_gi": int(main_gi),
            "support_gis": [int(x) for x in support_gis],
            "all_gis": [int(x) for x in gis],
            "layer": int(arr[main_gi, 2]),
            "t": float(tsum / wsum),
            "p": float(psum / wsum),
            "adc": int(adc[main_gi]),
            "kind": "horizontal",
        })

    # every leftover hit not in a horizontal chain becomes a 1-hit blob
    for gi in range(len(arr)):
        gi = int(gi)
        if gi in used_blocked:
            continue
        if gi in hits_in_hchains:
            continue

        blob_groups.append({
            "main_gi": gi,
            "support_gis": [],
            "all_gis": [gi],
            "layer": int(arr[gi, 2]),
            "t": float(arr[gi, 0]),
            "p": float(arr[gi, 1]),
            "adc": int(adc[gi]),
            "kind": "single",
        })

    # sort by ADC descending for seeding
    blob_groups.sort(key=lambda b: int(b["adc"]), reverse=True)

    return blob_groups, horiz_chains

def _consume_blob_group(key, blob, chain_main, chain_consumed, used_global):
    """
    Add whole blob group to the chain and mark all its hits used.
    """
    main_gi = int(blob["main_gi"])
    sup_gis = [int(x) for x in blob["support_gis"]]

    if main_gi in used_global:
        return False
    if any(s in used_global for s in sup_gis):
        return False

    used_global.add(main_gi)
    chain_main.append(main_gi)
    chain_consumed.append(main_gi)

    for s in sup_gis:
        used_global.add(s)
        chain_consumed.append(s)

    return True


def _blob_group_points_for_fit(key, blob):
    """
    Convert one blob group to fit points [(L,t,p,w), ...]
    """
    out = []
    all_gis = [int(blob["main_gi"])] + [int(x) for x in blob["support_gis"]]
    for gi in all_gis:
        out.append((
            float(pts_arr[key][gi][2]),
            float(pts_arr[key][gi][0]),
            float(pts_arr[key][gi][1]),
            float(adc_arr[key][gi]),
        ))
    return out


def _find_best_blob_group_near(blob_groups, used_global, layer, t_pred, p_pred, dt, dp):
    """
    Among blob groups on the target layer, find best unused one near prediction.
    """
    best = None
    best_score = None

    for blob in blob_groups:
        if int(blob["layer"]) != int(layer):
            continue

        all_gis = blob["all_gis"]
        if any(int(gi) in used_global for gi in all_gis):
            continue

        dtv = abs(float(blob["t"]) - float(t_pred))
        dpv = abs(float(blob["p"]) - float(p_pred))
        if dtv > dt or dpv > dp:
            continue

        score = (dtv*dtv + dpv*dpv, -int(blob["adc"]))
        if best_score is None or score < best_score:
            best = blob
            best_score = score

    return best


def build_vertical_chain_from_blob_groups(
    key,
    seed_blob,
    blob_groups,
    used_global,
    dt=6,
    dp=6,
    dl=1,
    max_skip=1
):
    """
    Second-pass vertical chain builder using blob groups instead of raw hit groups.
    """
    chain_main, chain_consumed = [], []
    fit_hist_hw, fit_hist_loc  = [], []

    ok = _consume_blob_group(key, seed_blob, chain_main, chain_consumed, used_global)
    if not ok:
        return chain_main, chain_consumed, fit_hist_hw, fit_hist_loc

    cur_blob = seed_blob
    l0 = int(seed_blob["layer"])

    # find first next-layer blob
    next_blob = None
    next_layer_actual = None
    for skip in range(max_skip + 1):
        test_layer = l0 + dl*(skip + 1)
        cand = _find_best_blob_group_near(
            blob_groups, used_global,
            layer=test_layer,
            t_pred=float(seed_blob["t"]),
            p_pred=float(seed_blob["p"]),
            dt=dt, dp=dp
        )
        if cand is not None:
            next_blob = cand
            next_layer_actual = test_layer
            break

    if next_blob is None:
        return chain_main, chain_consumed, fit_hist_hw, fit_hist_loc

    ok = _consume_blob_group(key, next_blob, chain_main, chain_consumed, used_global)
    if not ok:
        return chain_main, chain_consumed, fit_hist_hw, fit_hist_loc

    # initial fit from first two blobs
    pts_hw = _blob_group_points_for_fit(key, seed_blob) + _blob_group_points_for_fit(key, next_blob)
    at, bt, ap, bp = _fit_from_pts(pts_hw)
    fit_hist_hw.append({"t_fit": {"a": at, "b": bt},
                        "p_fit": {"a": ap, "b": bp}})

    pts_loc = []
    for blob in [seed_blob, next_blob]:
        for gi in [int(blob["main_gi"])] + [int(x) for x in blob["support_gis"]]:
            pts_loc.append((
                float(local_coords_arr[key][gi][2]),
                float(local_coords_arr[key][gi][0]),
                float(local_coords_arr[key][gi][1]),
                float(adc_arr[key][gi]),
            ))
    at_loc, bt_loc, ap_loc, bp_loc = _fit_from_pts(pts_loc)
    fit_hist_loc.append({"t_fit": {"a": at_loc, "b": bt_loc},
                         "p_fit": {"a": ap_loc, "b": bp_loc}})

    cur_blob = next_blob
    cur_layer = int(next_layer_actual)
    misses = 0

    while True:
        target = cur_layer + dl
        L_min  = 16*key[1] + 7
        L_max  = 16*(key[1] + 1) + 7
        if target < L_min or target >= L_max:
            break

        found_blob = None
        found_layer = None

        for skip in range(max_skip + 1):
            sl = target + dl*skip
            if sl < L_min or sl >= L_max:
                continue

            t_pred = at + bt * float(sl)
            p_pred = ap + bp * float(sl)

            cand = _find_best_blob_group_near(
                blob_groups, used_global,
                layer=sl, t_pred=t_pred, p_pred=p_pred,
                dt=dt, dp=dp
            )
            if cand is not None:
                found_blob = cand
                found_layer = sl
                misses = 0
                break

        if found_blob is None:
            misses += 1
            if misses > max_skip:
                break
            cur_layer = target
            continue

        ok = _consume_blob_group(key, found_blob, chain_main, chain_consumed, used_global)
        if not ok:
            break

        pts_hw = _blob_group_points_for_fit(key, cur_blob) + _blob_group_points_for_fit(key, found_blob)
        at, bt, ap, bp = _fit_from_pts(pts_hw)
        fit_hist_hw.append({"t_fit": {"a": at, "b": bt},
                            "p_fit": {"a": ap, "b": bp}})

        pts_loc = []
        for blob in [cur_blob, found_blob]:
            for gi in [int(blob["main_gi"])] + [int(x) for x in blob["support_gis"]]:
                pts_loc.append((
                    float(local_coords_arr[key][gi][2]),
                    float(local_coords_arr[key][gi][0]),
                    float(local_coords_arr[key][gi][1]),
                    float(adc_arr[key][gi]),
                ))
        at_loc, bt_loc, ap_loc, bp_loc = _fit_from_pts(pts_loc)
        fit_hist_loc.append({"t_fit": {"a": at_loc, "b": bt_loc},
                             "p_fit": {"a": ap_loc, "b": bp_loc}})

        cur_blob = found_blob
        cur_layer = int(found_layer)

    return chain_main, chain_consumed, fit_hist_hw, fit_hist_loc

def find_vertical_chains_from_blob_groups(
    key,
    used_global,
    blob_groups,
    dt=6,
    dp=6,
    min_hits=3,
    chain_id_start=300000
):
    """
    Second-pass vertical chains using:
      - horizontal chains as blob groups
      - leftover single hits as 1-hit blobs
    """
    chains = []
    cid = int(chain_id_start)

    # seed by strongest blobs first
    seeds = sorted(blob_groups, key=lambda b: int(b["adc"]), reverse=True)

    for seed_blob in seeds:
        if any(int(gi) in used_global for gi in seed_blob["all_gis"]):
            continue

        cm, cc, fh_hw, fh_loc = build_vertical_chain_from_blob_groups(
            key, seed_blob, blob_groups, used_global,
            dt=dt, dp=dp, dl=1, max_skip=1
        )

        if len(cm) < min_hits:
            continue

        chain_rec = {
            "chain_id": cid,
            "key": key,
            "chain_main": [int(x) for x in cm],
            "chain_consumed": [int(x) for x in cc],
            "fit_history_hw": fh_hw,
            "fit_history_loc": fh_loc,
            "seed_kind": seed_blob["kind"],
        }

        chain_rec["sagitta_fit"] = fit_chain_with_sagitta(
            chain_rec, local_coords_arr, adc_arr=adc_arr
        )

        chains.append(chain_rec)
        cid += 1

    return chains




## Track Matching
Connect chains within a module, then across modules.

In [ ]:
import math
import copy
import numpy as np


def _sorted_chain_hits_by_r(chain, use_consumed=False):
    key = chain["key"]
    hit_list = chain["chain_consumed"] if use_consumed else chain["chain_main"]
    return sorted(hit_list, key=lambda gi: float(local_coords_arr[key][gi][2]))


def _chain_r_bounds(chain, use_consumed=False):
    hs = _sorted_chain_hits_by_r(chain, use_consumed=use_consumed)
    if not hs:
        return (0.0, 0.0)
    key = chain["key"]
    r0 = float(local_coords_arr[key][hs[0]][2])
    r1 = float(local_coords_arr[key][hs[-1]][2])
    return r0, r1


def _predict_chain_at_r(chain, r):
    fit = chain.get("sagitta_fit", None)
    if fit is None:
        return None
    return (
        float(fit["m_zr"])   * r + float(fit["c_zr"]),
        float(fit["m_phir"]) * r + float(fit["c_phir"]),
    )


def _endpoint_match_score(chain_a, chain_b,
                          r_gap_max=6.0,
                          t_tol=6.0,
                          phi_tol=0.02):
    """
    Decide whether two chains should be merged end-to-end.

    Returns:
        (ok, gap, dt, dphi, order)

    order = "ab" means a is inner and b is outer
          = "ba" means b is inner and a is outer
    """
    ra0, ra1 = _chain_r_bounds(chain_a, use_consumed=False)
    rb0, rb1 = _chain_r_bounds(chain_b, use_consumed=False)

    # determine radial ordering
    if ra1 <= rb0:
        r_gap = rb0 - ra1
        r_mid = 0.5 * (ra1 + rb0)
        pa = _predict_chain_at_r(chain_a, r_mid)
        pb = _predict_chain_at_r(chain_b, r_mid)
        order = "ab"
    elif rb1 <= ra0:
        r_gap = ra0 - rb1
        r_mid = 0.5 * (rb1 + ra0)
        pa = _predict_chain_at_r(chain_b, r_mid)
        pb = _predict_chain_at_r(chain_a, r_mid)
        order = "ba"
    else:
        # overlap in radius; still allow merge if fits agree well
        r_gap = 0.0
        r_mid = 0.5 * (max(ra0, rb0) + min(ra1, rb1))
        pa = _predict_chain_at_r(chain_a, r_mid)
        pb = _predict_chain_at_r(chain_b, r_mid)
        order = "overlap"

    if pa is None or pb is None:
        return (False, None, None, None, None)

    dt   = abs(pa[0] - pb[0])
    dphi = abs(wrap_pi(pa[1] - pb[1]))

    ok = (r_gap <= r_gap_max) and (dt <= t_tol) and (dphi <= phi_tol)
    return (ok, r_gap, dt, dphi, order)


def _merge_two_chains(chain_a, chain_b, new_chain_id=None):
    """
    Merge hit content only, then refit using your existing fitter.
    """
    merged = copy.deepcopy(chain_a)

    # unique hit lists
    main_set = set(int(x) for x in chain_a["chain_main"])
    main_set.update(int(x) for x in chain_b["chain_main"])

    consumed_set = set(int(x) for x in chain_a["chain_consumed"])
    consumed_set.update(int(x) for x in chain_b["chain_consumed"])

    sat_set = set(int(x) for x in chain_a.get("chain_satellites", []))
    sat_set.update(int(x) for x in chain_b.get("chain_satellites", []))

    key = chain_a["key"]

    merged["chain_main"] = sorted(main_set,
                                  key=lambda gi: float(local_coords_arr[key][gi][2]))
    merged["chain_consumed"] = sorted(consumed_set,
                                      key=lambda gi: float(local_coords_arr[key][gi][2]))
    merged["chain_satellites"] = sorted(sat_set,
                                        key=lambda gi: float(local_coords_arr[key][gi][2]))

    if new_chain_id is not None:
        merged["chain_id"] = int(new_chain_id)

    # keep fit histories only for bookkeeping
    merged["fit_history_hw"] = list(chain_a.get("fit_history_hw", [])) + \
                               list(chain_b.get("fit_history_hw", []))
    merged["fit_history_loc"] = list(chain_a.get("fit_history_loc", [])) + \
                                list(chain_b.get("fit_history_loc", []))

    merged["sagitta_fit"] = fit_chain_with_sagitta(
        merged, local_coords_arr, adc_arr=adc_arr
    )
    return merged


def merge_close_vertical_chains(vertical_chains,
                                r_gap_max=6.0,
                                t_tol=6.0,
                                phi_tol=0.02,
                                max_passes=10,
                                debug=True):
    """
    Greedy iterative merge of close vertical chains.
    """
    chains = list(vertical_chains)
    if len(chains) < 2:
        return chains

    changed = True
    ipass = 0

    while changed and ipass < max_passes:
        changed = False
        ipass += 1

        used = set()
        new_chains = []
        next_id = 0

        # sort by inner radius
        order = sorted(range(len(chains)),
                       key=lambda i: _chain_r_bounds(chains[i])[0])

        i = 0
        while i < len(order):
            ia = order[i]
            if ia in used:
                i += 1
                continue

            best_j = None
            best_score = None

            for j in range(i + 1, len(order)):
                ib = order[j]
                if ib in used:
                    continue
                if chains[ia]["key"] != chains[ib]["key"]:
                    continue

                ok, gap, dt, dphi, merge_order = _endpoint_match_score(
                    chains[ia], chains[ib],
                    r_gap_max=r_gap_max, t_tol=t_tol, phi_tol=phi_tol
                )
                if not ok:
                    continue

                # smaller is better
                score = (gap, dt, dphi)

                if best_score is None or score < best_score:
                    best_score = score
                    best_j = ib

            if best_j is not None:
                merged = _merge_two_chains(chains[ia], chains[best_j], new_chain_id=next_id)
                new_chains.append(merged)
                used.add(ia)
                used.add(best_j)
                changed = True

                if debug:
                    print("[merge_close_vertical_chains] merged "
                          "chain %d + %d -> new chain %d | "
                          "gap=%.2f dt=%.2f dphi=%.4f" %
                          (chains[ia]["chain_id"], chains[best_j]["chain_id"], next_id,
                           best_score[0], best_score[1], best_score[2]))
                next_id += 1
            else:
                ch = copy.deepcopy(chains[ia])
                ch["chain_id"] = next_id
                new_chains.append(ch)
                used.add(ia)
                next_id += 1

            i += 1

        chains = new_chains

    if debug:
        print("After chain merging: %d chains" % len(chains))

    return chains

In [ ]:



def _chain_r_extent(chain):
    """Return (r_min, r_max) of the chain's main hits in local coordinates."""
    key   = chain["key"]
    radii = [local_coords_arr[key][gi][2] for gi in chain["chain_main"]]
    return (min(radii), max(radii)) if radii else (0.0, 0.0)
def _track_r_extent(track):
    """Return (r_min, r_max) across all consumed hits of a module-track."""
    radii = []
    for ch in track["chains"]:
        key = ch["key"]
        for gi in ch["chain_consumed"]:
            radii.append(local_coords_arr[key][gi][2])
    return (min(radii), max(radii)) if radii else (0.0, 0.0)
def _gap_mid_radius(r_extent_a, r_extent_b):
    """
    Midpoint between the closest radial edges of two objects.
    Works whether they overlap slightly or are separated.
    """
    r_max_inner = min(max(r_extent_a), max(r_extent_b))
    r_min_outer = max(min(r_extent_a), min(r_extent_b))
    return 0.5 * (r_max_inner + r_min_outer)

def predict_at_radius(fit, r_target):
    """
    Predict matching quantities at radius r_target using local projected fits.

    Returns:
        (tbin_mid, phi_local_mid, theta_mid, alpha_mid)

    where
        tbin_mid      = value from tbin(r) fit
        phi_local_mid = value from local phi(r) fit
        theta_mid     = tangent angle of tbin(r) fit
        alpha_mid     = tangent angle of phi(r) fit
    """
    if fit is None:
        print("No fit available for prediction")
        return None

    r_target = float(r_target)

    # tbin vs r
    m_zr = fit.get("m_zr", None)
    c_zr = fit.get("c_zr", None)

    # local phi vs r
    m_phir = fit.get("m_phir", None)
    c_phir = fit.get("c_phir", None)

    if m_zr is None or c_zr is None or m_phir is None or c_phir is None:
        print("Fit parameters missing for prediction")
        return None

    tbin_mid = m_zr * r_target + c_zr
    phi_mid  = m_phir * r_target + c_phir

    # tangent angles of the projected lines
    theta_mid = math.atan(m_zr)
    alpha_mid = math.atan(m_phir)

    return tbin_mid, phi_mid, theta_mid, alpha_mid





def connect_within_module(vertical_chains,
                          z_window=3.0, theta_window=0.2,
                          phi_window=0.02, alpha_window=0.2,
                          debug=True):
    """
    Group chains from one module into module-tracks using the same greedy,
    iterative logic as connect_modules, with analogous debug printouts.
    """
    if not vertical_chains:
        return []

    key  = vertical_chains[0]["key"]
    imod = key[1]

    assigned = set()
    tracks   = []
    tid      = 0

    sorted_chains = sorted(
        vertical_chains,
        key=lambda ch: (-len(ch["chain_main"]),
                        ch["sagitta_fit"]["chi2_sag"] / max(ch["sagitta_fit"]["ndof_sag"], 1)
                        if ch["sagitta_fit"] else float("inf"))
    )

    extents = {ch["chain_id"]: _chain_r_extent(ch) for ch in sorted_chains}
    
    def group_extent(group):
        radii = []
        for ch in group:
            ch_key = ch["key"]
            for gi in ch["chain_consumed"]:
                radii.append(float(local_coords_arr[ch_key][gi][2]))
        return (min(radii), max(radii)) if radii else (0.0, 0.0)
    if debug:
        print(f"Connecting chains within module {imod}...")
        for ch in sorted_chains:
            fit = ch.get("sagitta_fit")
            chi2_ndof = (fit["chi2_sag"] / fit["ndof_sag"]) if fit else float("inf")
            print(f"  chain_id={ch['chain_id']} n_main={len(ch['chain_main'])} "
                  f"chi2/ndof={chi2_ndof:.3f} r_extent=({extents[ch['chain_id']][0]:.3f}, "
                  f"{extents[ch['chain_id']][1]:.3f})")

    def compatible(group, group_fit, group_ext, cand):
        if debug:
            print("---- connect_within_module candidate check ----")
            print(f"group_chain_ids = {[ch['chain_id'] for ch in group]}")
            print(f"candidate_chain_id = {cand['chain_id']}")

        if group_fit is None:
            if debug:
                print("result: REJECT early because group_fit is None\n")
            return None

        if cand.get("sagitta_fit") is None:
            if debug:
                print("result: REJECT early because candidate sagitta_fit is None\n")
            return None

        cand_ext = extents[cand["chain_id"]]
        r_mid    = _gap_mid_radius(group_ext, cand_ext)

        if debug:
            print(f"r_mid = {r_mid:.3f}")
            print(f"group_ext = ({group_ext[0]:.3f}, {group_ext[1]:.3f})")
            print(f"cand_ext  = ({cand_ext[0]:.3f}, {cand_ext[1]:.3f})")

        pred_g = predict_at_radius(group_fit, r_mid)
        pred_c = predict_at_radius(cand["sagitta_fit"], r_mid)

        if pred_g is None:
            if debug:
                print("result: REJECT early because predict_at_radius(group_fit, r_mid) returned None\n")
            return None

        if pred_c is None:
            if debug:
                print("result: REJECT early because predict_at_radius(cand['sagitta_fit'], r_mid) returned None\n")
            return None

        tbin_g, phi_g, theta_g, alpha_g = pred_g
        tbin_c, phi_c, theta_c, alpha_c = pred_c

        dtbin  = abs(tbin_g - tbin_c)
        dphi   = abs(phi_g - phi_c)
        dtheta = abs(wrap_pi(theta_g - theta_c))
        dalpha = abs(wrap_pi(alpha_g - alpha_c))

        if debug:
            print(
                f"current/group chain: module={imod} "
                f"chain_ids={[ch['chain_id'] for ch in group]} "
                f"phi_local={phi_g:.6f} tbin={tbin_g:.3f} "
                f"alpha={alpha_g:.6f} theta={theta_g:.6f}"
            )
            print(
                f"candidate chain    : module={imod} chain_id={cand['chain_id']} "
                f"phi_local={phi_c:.6f} tbin={tbin_c:.3f} "
                f"alpha={alpha_c:.6f} theta={theta_c:.6f}"
            )
            print(
                f"diffs: dtbin={dtbin:.3f} dphi={dphi:.6f} "
                f"dalpha={dalpha:.6f} dtheta={dtheta:.6f}"
            )

        if dtbin > z_window or dtheta > theta_window or dphi > phi_window or dalpha > alpha_window:
            if debug:
                failed = []
                if dtbin > z_window:
                    failed.append(f"dtbin ({dtbin:.3f} > {z_window:.3f})")
                if dphi > phi_window:
                    failed.append(f"dphi ({dphi:.6f} > {phi_window:.6f})")
                if dalpha > alpha_window:
                    failed.append(f"dalpha ({dalpha:.6f} > {alpha_window:.6f})")
                if dtheta > theta_window:
                    failed.append(f"dtheta ({dtheta:.6f} > {theta_window:.6f})")
                print("result: REJECT by " + ", ".join(failed) + "\n")
            return None

        dist = math.sqrt(
            (dtbin  / max(z_window,     1e-12)) ** 2 +
            (dphi   / max(phi_window,   1e-12)) ** 2 +
            (dtheta / max(theta_window, 1e-12)) ** 2 +
            (dalpha / max(alpha_window, 1e-12)) ** 2
        )

        if debug:
            print(f"result: PASS dist={dist:.6f}\n")

        return {
            "r_mid": r_mid,
            "dtbin": dtbin,
            "dtheta": dtheta,
            "dphi": dphi,
            "dalpha": dalpha,
            "dist": dist,
        }

    for seed in sorted_chains:
        if seed["chain_id"] in assigned:
            continue

        if seed["sagitta_fit"] is None:
            assigned.add(seed["chain_id"])
            tracks.append({
                "track_id": tid,
                "chain_ids": [seed["chain_id"]],
                "chains": [seed],
                "key": key,
                "n_chains": 1,
            })
            tid += 1
            continue

        group = [seed]
        assigned.add(seed["chain_id"])
        group_fit = seed["sagitta_fit"]
        group_ext = extents[seed["chain_id"]]

        grew = True
        while grew:
            grew = False
            best_cand = None
            best_info = None

            for cand in sorted_chains:
                if cand["chain_id"] in assigned:
                    continue
                if cand["sagitta_fit"] is None:
                    continue

                info = compatible(group, group_fit, group_ext, cand)
                if info is None:
                    continue

                if best_info is None or info["dist"] < best_info["dist"]:
                    best_cand = cand
                    best_info = info

            if best_cand is not None:
                group.append(best_cand)
                assigned.add(best_cand["chain_id"])

                tmp_track = {
                    "track_id": -1,
                    "chains": list(group),
                    "key": key,
                    "n_chains": len(group),
                }
                refit_collection([tmp_track], local_coords_arr, adc_arr)

                if tmp_track.get("refit") is not None:
                    group_fit = tmp_track["refit"]
                    group_ext = group_extent(group)
                    grew = True
                else:
                    assigned.remove(best_cand["chain_id"])
                    group.pop()

        tracks.append({
            "track_id": tid,
            "chain_ids": [c["chain_id"] for c in group],
            "chains": group,
            "key": key,
            "n_chains": len(group),
        })
        tid += 1

    refit_collection(tracks, local_coords_arr, adc_arr)

    if debug:
        print(f"Module {imod}: {len(vertical_chains)} chains -> {len(tracks)} module-tracks")

    return tracks


def connect_modules(tracks_per_module,
                    tbin_window=5.0, theta_window=0.2,
                    phi_window=0.05, alpha_window=0.2,
                    debug=True):
    """
    Stitch module-tracks into full tracks using projected local matching:
      * tbin at r_mid from tbin(r)
      * phi at r_mid from local phi(r)
      * theta = tangent angle of tbin(r)
      * alpha = tangent angle of phi(r)
    """
    flat_tracks = []
    for imod in sorted(tracks_per_module):
        for mt in tracks_per_module[imod]:
            rec = dict(mt)
            rec["module"] = int(imod)
            rec["uid"] = (int(imod), int(mt["track_id"]))
            flat_tracks.append(rec)

    if not flat_tracks:
        return []

    assigned = set()
    tracks   = []
    tid      = 0

    sorted_tracks = sorted(
        flat_tracks,
        key=lambda mt: (-sum(len(ch["chain_main"]) for ch in mt["chains"]),
                        mt["refit"]["chi2_sag"] / max(mt["refit"]["ndof_sag"], 1)
                        if mt.get("refit") is not None else float("inf"),
                        mt["module"])
    )

    def group_extent(group):
        radii = []
        for mt in group:
            for ch in mt["chains"]:
                key = ch["key"]
                for gi in ch["chain_consumed"]:
                    radii.append(float(local_coords_arr[key][gi][2]))
        return (min(radii), max(radii)) if radii else (0.0, 0.0)

    def compatible(group, group_fit, group_ext, cand):
        if group_fit is None or cand.get("refit") is None:
            return None

        group_modules = {mt["module"] for mt in group}
        if cand["module"] in group_modules:
            return None

        cand_ext = _track_r_extent(cand)
        r_mid    = _gap_mid_radius(group_ext, cand_ext)

        pred_g = predict_at_radius(group_fit, r_mid)
        pred_c = predict_at_radius(cand["refit"], r_mid)
        if pred_g is None or pred_c is None:
            return None

        tbin_g, phi_g, theta_g, alpha_g = pred_g
        tbin_c, phi_c, theta_c, alpha_c = pred_c

        dtbin  = abs(tbin_g - tbin_c)
        dphi   = abs(phi_g - phi_c)
        dtheta = abs(wrap_pi(theta_g - theta_c))
        dalpha = abs(wrap_pi(alpha_g - alpha_c))

        if debug:
            print("---- connect_modules candidate check ----")
            print(f"r_mid = {r_mid:.3f}")
            print(
                f"current/group track: modules={sorted(group_modules)} "
                f"uid_list={[mt['uid'] for mt in group]} "
                f"phi_local={phi_g:.6f} tbin={tbin_g:.3f} "
                f"alpha={alpha_g:.6f} theta={theta_g:.6f}"
            )
            print(
                f"candidate track    : module={cand['module']} uid={cand['uid']} "
                f"phi_local={phi_c:.6f} tbin={tbin_c:.3f} "
                f"alpha={alpha_c:.6f} theta={theta_c:.6f}"
            )
            print(
                f"diffs: dtbin={dtbin:.3f} dphi={dphi:.6f} "
                f"dalpha={dalpha:.6f} dtheta={dtheta:.6f}"
            )

        if dtbin > tbin_window or dtheta > theta_window or dphi > phi_window or dalpha > alpha_window:
            if debug:
                failed = []
                if dtbin > tbin_window:
                    failed.append(f"dtbin ({dtbin:.3f} > {tbin_window:.3f})")
                if dphi > phi_window:
                    failed.append(f"dphi ({dphi:.6f} > {phi_window:.6f})")
                if dalpha > alpha_window:
                    failed.append(f"dalpha ({dalpha:.6f} > {alpha_window:.6f})")
                if dtheta > theta_window:
                    failed.append(f"dtheta ({dtheta:.6f} > {theta_window:.6f})")
                print("result: REJECT by " + ", ".join(failed) + "\n")
            return None

        dist = math.sqrt(
            (dtbin / max(tbin_window, 1e-12)) ** 2 +
            (dphi  / max(phi_window,  1e-12)) ** 2 +
            (dtheta / max(theta_window, 1e-12)) ** 2 +
            (dalpha / max(alpha_window, 1e-12)) ** 2
        )

        if debug:
            print(f"result: PASS dist={dist:.6f}\n")

        return {
            "r_mid": r_mid,
            "dtbin": dtbin,
            "dtheta": dtheta,
            "dphi": dphi,
            "dalpha": dalpha,
            "dist": dist,
        }

    for seed in sorted_tracks:
        if seed["uid"] in assigned:
            continue

        group = [seed]
        assigned.add(seed["uid"])
        group_fit = seed.get("refit")
        group_ext = _track_r_extent(seed)

        if group_fit is not None:
            grew = True
            while grew:
                grew = False
                best_cand = None
                best_info = None

                for cand in sorted_tracks:
                    if cand["uid"] in assigned:
                        continue

                    info = compatible(group, group_fit, group_ext, cand)
                    if info is None:
                        continue

                    if best_info is None or info["dist"] < best_info["dist"]:
                        best_cand = cand
                        best_info = info

                if best_cand is not None:
                    group.append(best_cand)
                    assigned.add(best_cand["uid"])

                    tmp_track = {
                        "track_id": -1,
                        "chains": [ch for mt in group for ch in mt["chains"]],
                    }
                    refit_collection([tmp_track], local_coords_arr, adc_arr)

                    if tmp_track.get("refit") is not None:
                        group_fit = tmp_track["refit"]
                        group_ext = group_extent(group)
                        grew = True
                    else:
                        assigned.remove(best_cand["uid"])
                        group.pop()

        all_chains = [ch for mt in group for ch in mt["chains"]]
        tracks.append({
            "track_id": tid,
            "module_track_ids": [mt["uid"] for mt in group],
            "chains": all_chains,
            "chain_ids": [c["chain_id"] for c in all_chains],
            "modules": sorted({mt["module"] for mt in group}),
            "n_chains": len(all_chains),
        })
        tid += 1

    refit_collection(tracks, local_coords_arr, adc_arr)

    if debug:
        print(f"Connected full tracks: {len(tracks)}")

    return tracks

## Reconstruction

In [ ]:
sector = 0
side   = 0

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def collect_used_hits_from_full_tracks(full_tracks):
    """
    Return dict: imod -> set(global hit indices consumed by final full tracks)
    """
    used_per_module = {0: set(), 1: set(), 2: set()}

    for trk in full_tracks:
        for ch in trk["chains"]:
            key = ch["key"]
            imod = key[1]
            for gi in ch["chain_consumed"]:
                used_per_module[imod].add(gi)

    return used_per_module


def merge_used_hits(base_used, extra_used):
    out = {}
    all_keys = set(base_used.keys()) | set(extra_used.keys())
    for k in all_keys:
        out[k] = set(base_used.get(k, set())) | set(extra_used.get(k, set()))
    return out


# ------------------------------------------------------------
# First pass
# ------------------------------------------------------------
vchains_per_module_1 = {}
tracks_per_module_1  = {}

for imod in range(3):
    key         = (sector, imod, side)
    used_global = set()

    # pre-consume noisy clusters
    find_horizontal_chains(
        key, used_global,
        dt=2, dp=2, min_hits=3, min_pad_span=5, chain_id_start=0
    )

    vertical_chains = find_vertical_chains(key, used_global)
    attach_satellites(key, vertical_chains, used_global)
    vertical_chains = merge_close_vertical_chains(
        vertical_chains,
        r_gap_max=6.0,   # how far apart in radius the chain ends may be
        t_tol=6.0,       # allowed difference in tbin(z) prediction
        phi_tol=0.02,    # allowed difference in local phi prediction
        debug=True
        )

    vchains_per_module_1[imod] = vertical_chains

    module_tracks = connect_within_module(
        vertical_chains,
        z_window=10.0, phi_window=0.02, theta_window=0.3, alpha_window=0.5,
        debug=False
    )
    tracks_per_module_1[imod] = module_tracks

    n_hits = sum(len(c["chain_consumed"]) for c in vertical_chains)
    print(f"[pass1] Module {imod}: {len(vertical_chains)} chains, {n_hits} hits, "
          f"{len(module_tracks)} module-tracks")

full_tracks_1 = connect_modules(
    tracks_per_module_1,
    tbin_window=10.0, phi_window=0.02, theta_window=0.3, alpha_window=0.5,
    debug=False
)
full_tracks_1 = grow_full_tracks_with_tube_hits(
    full_tracks_1,
    sector=sector,
    side=side,
    t_window=3.0,
    phi_window=0.010,
    r_margin=2.0,
    max_iterations=2,
    debug=True
)
total_vchains_1 = sum(len(v) for v in vchains_per_module_1.values())
total_hits_1    = sum(len(c["chain_consumed"])
                      for v in vchains_per_module_1.values() for c in v)

print(f"\n{'='*50}")
print(f"[pass1] Vertical chains  : {total_vchains_1}  ({total_hits_1} hits)")
print(f"[pass1] Module tracks    : {sum(len(m) for m in tracks_per_module_1.values())}")
print(f"[pass1] Full tracks      : {len(full_tracks_1)}")
print(f"{'='*50}")

# ------------------------------------------------------------
# Build leftover-hit masks from final first-pass full tracks
# ------------------------------------------------------------
used_by_fulltracks_1 = collect_used_hits_from_full_tracks(full_tracks_1)

# ------------------------------------------------------------
# Second pass on leftover hits
#   - looser vertical-chain windows: dp=6, dt=6
#   - same module/full-track connection logic
# ------------------------------------------------------------
vchains_per_module_2 = {}
tracks_per_module_2  = {}
hchains_per_module_2 = {}
blob_groups_per_module_2 = {}

for imod in range(3):
    key = (sector, imod, side)

    # block only hits already used by pass1 full tracks
    used_global = set(used_by_fulltracks_1.get(imod, set()))

    # build pass2 blob groups from leftovers
    blob_groups, horiz_chains_2 = build_blob_groups_from_horizontal_chains(
        key,
        used_blocked=used_global,
        h_dt=2,
        h_dp=2,
        h_min_hits=3,
        h_min_pad_span=5,
        chain_id_start=100000
    )

    hchains_per_module_2[imod] = horiz_chains_2
    blob_groups_per_module_2[imod] = blob_groups

    # now actually consume blob groups while building vertical chains
    vertical_chains_2 = find_vertical_chains_from_blob_groups(
        key,
        used_global=used_global,
        blob_groups=blob_groups,
        dt=6,
        dp=6,
        min_hits=3,
        chain_id_start=200000
    )

    attach_satellites(key, vertical_chains_2, used_global)

    vertical_chains_2 = merge_close_vertical_chains(
        vertical_chains_2,
        r_gap_max=6.0,
        t_tol=6.0,
        phi_tol=0.02,
        debug=True
    )

    vchains_per_module_2[imod] = vertical_chains_2

    module_tracks_2 = connect_within_module(
        vertical_chains_2,
        z_window=10.0, phi_window=0.02, theta_window=0.3, alpha_window=0.5,
        debug=False
    )
    tracks_per_module_2[imod] = module_tracks_2

    n_hits_2 = sum(len(c["chain_consumed"]) for c in vertical_chains_2)
    print(f"[pass2] Module {imod}: "
          f"{len(horiz_chains_2)} horiz-blobs, "
          f"{len(blob_groups)} total blobs, "
          f"{len(vertical_chains_2)} vchains, "
          f"{n_hits_2} hits, "
          f"{len(module_tracks_2)} module-tracks")
    
full_tracks_2 = connect_modules(
    tracks_per_module_2,
    tbin_window=10.0, phi_window=0.02, theta_window=0.3, alpha_window=0.5,
    debug=False
)

total_vchains_2 = sum(len(v) for v in vchains_per_module_2.values())
total_hits_2    = sum(len(c["chain_consumed"])
                      for v in vchains_per_module_2.values() for c in v)



print(f"\n{'='*50}")
print(f"[pass2] Vertical chains  : {total_vchains_2}  ({total_hits_2} hits)")
print(f"[pass2] Module tracks    : {sum(len(m) for m in tracks_per_module_2.values())}")
print(f"[pass2] Full tracks      : {len(full_tracks_2)}")
print(f"{'='*50}")

# ------------------------------------------------------------
# Final combined result
# ------------------------------------------------------------
full_tracks_all = list(full_tracks_1) + list(full_tracks_2)

print(f"\n{'='*50}")
print(f"Combined full tracks : {len(full_tracks_all)}")
print(f"  pass1 = {len(full_tracks_1)}")
print(f"  pass2 = {len(full_tracks_2)}")
print(f"{'='*50}")

vchains_per_module = {}
tracks_per_module = {}

for imod in range(3):
    vchains_per_module[imod] = list(vchains_per_module_1.get(imod, [])) + list(vchains_per_module_2.get(imod, []))
    tracks_per_module[imod] = list(tracks_per_module_1.get(imod, [])) + list(tracks_per_module_2.get(imod, []))

full_tracks = full_tracks_all

## Clustering

In [ ]:
# ── Cluster positions on final full tracks ────────────────────────────────────

def build_track_layer_clusters(full_tracks, adc_power=None, adc_min=1.0):
    """
    For each full track, group all consumed hits by detector layer and compute
    one ADC-weighted cluster centroid per layer.

    Returns:
        clusters_per_track: list of dicts
            [
              {
                "track_id": ...,
                "clusters": [
                    {
                      "layer": int,
                      "imod": int,
                      "nhits": int,
                      "adc_sum": float,
                      "tbin": float,        # ADC-weighted centroid
                      "phi": float,         # ADC-weighted centroid (local phi)
                      "radius": float,
                      "pad": float,         # ADC-weighted centroid in pad units
                      "hit_indices": [...], # global indices within that key
                      "key": (sector, imod, side)
                    }, ...
                ]
              }, ...
            ]
    """
    clusters_per_track = []

    for trk in full_tracks:
        # collect hits by absolute layer
        by_layer = {}

        for ch in trk["chains"]:
            key = ch["key"]
            imod = key[1]

            for gi in ch["chain_consumed"]:
                gi = int(gi)

                tbin_hw, pad_hw, layer = pts_arr[key][gi]
                tbin_loc, phi_loc, radius = local_coords_arr[key][gi]
                adc = float(adc_arr[key][gi])

                layer = int(layer)
                if layer not in by_layer:
                    by_layer[layer] = {
                        "layer": layer,
                        "imod": imod,
                        "key": key,
                        "hits": []
                    }

                by_layer[layer]["hits"].append({
                    "gi": gi,
                    "adc": max(adc, adc_min),
                    "tbin": float(tbin_loc),
                    "phi": float(phi_loc),
                    "radius": float(radius),
                    "pad": float(pad_hw),
                })

        track_clusters = []
        for layer in sorted(by_layer.keys()):
            entry = by_layer[layer]
            hits = entry["hits"]

            w = np.array([h["adc"] for h in hits], dtype=float)
            if adc_power is not None:
                w = np.power(w, float(adc_power))
            wsum = float(np.sum(w))
            if wsum <= 0:
                continue

            tbar = float(np.sum(w * np.array([h["tbin"] for h in hits])) / wsum)
            phibar = float(np.sum(w * np.array([h["phi"] for h in hits])) / wsum)
            padbar = float(np.sum(w * np.array([h["pad"] for h in hits])) / wsum)

            # same layer => same radius, but average anyway for safety
            rbar = float(np.sum(w * np.array([h["radius"] for h in hits])) / wsum)

            track_clusters.append({
                "layer": entry["layer"],
                "imod": entry["imod"],
                "nhits": len(hits),
                "adc_sum": wsum,
                "tbin": tbar,
                "phi": phibar,
                "radius": rbar,
                "pad": padbar,
                "hit_indices": [h["gi"] for h in hits],
                "key": entry["key"],
            })

        clusters_per_track.append({
            "track_id": trk["track_id"],
            "clusters": track_clusters
        })

    return clusters_per_track


In [ ]:
import numpy as np
import ROOT as root

# ------------------------------------------------------------
# SAMPA-like shaping kernel
# ------------------------------------------------------------
def make_sampa_kernel(nt=12, tau_rise=1.0, tau_fall=3.0, normalize=True):
    """
    Simple positive shaping kernel in time-bin units.
    Tune tau_rise / tau_fall to your measured SAMPA pulse shape.
    """
    t = np.arange(nt, dtype=float)
    h = (1.0 - np.exp(-t / max(tau_rise, 1e-6))) * np.exp(-t / max(tau_fall, 1e-6))
    h[0] = max(h[0], 1e-12)

    if normalize:
        s = np.sum(h)
        if s > 0:
            h = h / s
    return h


# ------------------------------------------------------------
# Positive 1D deconvolution
# ------------------------------------------------------------
def richardson_lucy_1d(signal, kernel, n_iter=25, eps=1e-9):
    """
    Positive deconvolution of one waveform.
    """
    y = np.asarray(signal, dtype=float).copy()
    y[y < 0] = 0.0

    h = np.asarray(kernel, dtype=float).copy()
    h[h < 0] = 0.0
    h = h / max(np.sum(h), eps)

    x = np.maximum(y.copy(), eps)
    h_rev = h[::-1]

    for _ in range(int(n_iter)):
        conv_x = np.convolve(x, h, mode="full")[:len(y)]
        ratio = y / np.maximum(conv_x, eps)
        corr = np.convolve(ratio, h_rev, mode="full")[:len(y)]
        x *= corr
        x[x < 0] = 0.0

    return x


# ------------------------------------------------------------
# Build 2D charge map for one cluster, no pedestal subtraction
# ------------------------------------------------------------
def make_cluster_charge_map_no_pedestal(cluster_hits):
    """
    cluster_hits: list of dicts with keys:
        gi, adc, tbin, phi, radius, pad
    """
    pads  = np.array([h["pad"]  for h in cluster_hits], dtype=float)
    tbins = np.array([h["tbin"] for h in cluster_hits], dtype=float)

    pad_bins  = np.array([int(round(x)) for x in pads], dtype=int)
    tbin_bins = np.array([int(round(x)) for x in tbins], dtype=int)

    upads  = np.sort(np.unique(pad_bins))
    utbins = np.sort(np.unique(tbin_bins))

    pad_to_i  = {p:i for i,p in enumerate(upads)}
    tbin_to_j = {t:j for j,t in enumerate(utbins)}

    qmap    = np.zeros((len(upads), len(utbins)), dtype=float)
    phi_sum = np.zeros((len(upads), len(utbins)), dtype=float)
    adc_sum = np.zeros((len(upads), len(utbins)), dtype=float)

    for h in cluster_hits:
        p = int(round(h["pad"]))
        t = int(round(h["tbin"]))
        i = pad_to_i[p]
        j = tbin_to_j[t]

        adc = float(h["adc"])
        qmap[i, j] += adc
        phi_sum[i, j] += adc * float(h["phi"])
        adc_sum[i, j] += adc

    phi_map = np.zeros_like(qmap)
    mask = adc_sum > 0
    phi_map[mask] = phi_sum[mask] / adc_sum[mask]

    return {
        "qmap": qmap,
        "phi_map": phi_map,
        "upads": upads,
        "utbins": utbins,
    }


# ------------------------------------------------------------
# Unfold cluster along time axis only
# ------------------------------------------------------------
def unfold_cluster_charge_map_sampa(cluster_map, kernel=None, rl_iter=25):
    """
    For each pad, deconvolve the waveform in time.
    """
    qraw = cluster_map["qmap"]
    if kernel is None:
        kernel = make_sampa_kernel()

    qcorr = np.zeros_like(qraw)

    for i in range(qraw.shape[0]):
        trace = qraw[i, :]
        if np.sum(trace) <= 0:
            continue
        qcorr[i, :] = richardson_lucy_1d(trace, kernel, n_iter=rl_iter)

    return qcorr


# ------------------------------------------------------------
# Corrected centroid from unfolded charge
# ------------------------------------------------------------
def centroid_from_unfolded_charge(cluster_map, qcorr):
    utbins  = cluster_map["utbins"]
    phi_map = cluster_map["phi_map"]

    wsum = float(np.sum(qcorr))
    if wsum <= 0:
        return None

    t_grid = np.tile(utbins.reshape(1, -1), (qcorr.shape[0], 1)).astype(float)

    phi_use = phi_map.copy()
    for i in range(phi_use.shape[0]):
        row = phi_use[i, :]
        nz = row != 0
        if np.any(nz):
            fill = float(np.mean(row[nz]))
        else:
            fill = 0.0
        row[~nz] = fill
        phi_use[i, :] = row

    t_corr   = float(np.sum(qcorr * t_grid) / wsum)
    phi_corr = float(np.sum(qcorr * phi_use) / wsum)

    return {
        "tbin": t_corr,
        "phi": phi_corr,
        "adc_sum": wsum,
    }

def build_track_layer_clusters_corrected(full_tracks,
                                         adc_power=None,
                                         adc_min=1.0,
                                         sampa_kernel=None,
                                         rl_iter=25):
    """
    Build one cluster per layer per full track.
    Store both raw centroid and corrected centroid, where corrected means:
      - unfold time tail with shaping response
      - recompute centroid from unfolded charge
    """
    if sampa_kernel is None:
        sampa_kernel = make_sampa_kernel()

    clusters_per_track = []

    for trk in full_tracks:
        by_layer = {}

        for ch in trk["chains"]:
            key = ch["key"]
            imod = key[1]

            for gi in ch["chain_consumed"]:
                gi = int(gi)

                tbin_hw, pad_hw, layer = pts_arr[key][gi]
                tbin_loc, phi_loc, radius = local_coords_arr[key][gi]
                adc = float(adc_arr[key][gi])

                layer = int(layer)
                if layer not in by_layer:
                    by_layer[layer] = {
                        "layer": layer,
                        "imod": imod,
                        "key": key,
                        "hits": []
                    }

                by_layer[layer]["hits"].append({
                    "gi": gi,
                    "adc": max(adc, adc_min),
                    "tbin": float(tbin_loc),
                    "phi": float(phi_loc),
                    "radius": float(radius),
                    "pad": float(pad_hw),
                })

        track_clusters = []

        for layer in sorted(by_layer.keys()):
            entry = by_layer[layer]
            hits = entry["hits"]

            w = np.array([h["adc"] for h in hits], dtype=float)
            if adc_power is not None:
                w = np.power(w, float(adc_power))
            wsum = float(np.sum(w))
            if wsum <= 0:
                continue

            # raw centroid
            tbar   = float(np.sum(w * np.array([h["tbin"]   for h in hits])) / wsum)
            phibar = float(np.sum(w * np.array([h["phi"]    for h in hits])) / wsum)
            padbar = float(np.sum(w * np.array([h["pad"]    for h in hits])) / wsum)
            rbar   = float(np.sum(w * np.array([h["radius"] for h in hits])) / wsum)

            cluster = {
                "layer": entry["layer"],
                "imod": entry["imod"],
                "nhits": len(hits),
                "adc_sum": wsum,
                "tbin": tbar,
                "phi": phibar,
                "radius": rbar,
                "pad": padbar,
                "hit_indices": [h["gi"] for h in hits],
                "key": entry["key"],
                "cluster_corrected": None,
            }

            # corrected centroid from unfolded charge
            cmap  = make_cluster_charge_map_no_pedestal(hits)
            qcorr = unfold_cluster_charge_map_sampa(
                cmap,
                kernel=sampa_kernel,
                rl_iter=rl_iter
            )
            cc = centroid_from_unfolded_charge(cmap, qcorr)

            if cc is not None:
                cluster["cluster_corrected"] = {
                    "tbin": cc["tbin"],
                    "phi": cc["phi"],
                    "adc_sum": cc["adc_sum"],
                    "radius": rbar,
                }

            track_clusters.append(cluster)

        clusters_per_track.append({
            "track_id": trk["track_id"],
            "clusters": track_clusters
        })

    return clusters_per_track

In [ ]:

track_layer_clusters = build_track_layer_clusters(
    full_tracks,
    adc_power=None   # use raw ADC weights
    # adc_power=0.5  # optional: softer weighting if very large ADC dominates
)

'''for tc in track_layer_clusters[:3]:
    print("track", tc["track_id"])
    for cl in tc["clusters"]:
        print("  layer=%2d  nhits=%2d  tbin=%.2f  phi=%.5f  r=%.3f  adc_sum=%.1f" %
              (cl["layer"], cl["nhits"], cl["tbin"], cl["phi"], cl["radius"], cl["adc_sum"]))'''

sampa_kernel = make_sampa_kernel(
    nt=12,
    tau_rise=1.0,
    tau_fall=3.0
)

track_layer_clusters = build_track_layer_clusters_corrected(
    full_tracks,
    adc_power=None,
    adc_min=1.0,
    sampa_kernel=sampa_kernel,
    rl_iter=20
)

## Drawing
Toggle the flags below to show/hide each layer of the display.

In [ ]:
Draw_all_hits        = True   # background: all raw hits
Draw_chains          = True   # vertical chain fits
Draw_inmodule_tracks = True   # within-module matched tracks
Draw_full_tracks     = True   # cross-module full tracks
Draw_chain_residuals = True  # r·Δφ residual histogram

_draw_objs = []   # keep ROOT objects alive

TRACK_COLORS = [
    root.kRed, root.kBlue, root.kGreen+2, root.kMagenta,
    root.kCyan+1, root.kOrange+7, root.kYellow+2, root.kPink+7,
]

In [ ]:
# ── Histogram factory ─────────────────────────────────────────────────────────

def make_hist(name, title):
    """Create a TH3F in local (tbin, phi, radius) coordinates."""
    Ncanon = Npads[2]
    return root.TH3F(
        name, f"{title}; timebin; phi; radius",
        300, -0.5, 299.5,
        Ncanon, 0, Ncanon * phi_bin_width[2],
        int((module_radius[2][15] - module_radius[0][0]) / 0.5),
        module_radius[0][0], module_radius[2][15])

def fill_all_hits(hist, sector, side):
    """Fill hist with all hits for the given sector and side."""
    for imod in range(3):
        key = (sector, imod, side)
        for j in range(len(pts_arr[key])):
            tbin, phi, radius = local_coords_arr[key][j]
            hist.Fill(tbin, phi, radius, int(adc_arr[key][j]))

# ── Sagitta polyline builder ──────────────────────────────────────────────────

def make_sagitta_polyline(name, fit, x_hits, y_hits,
                          npoints=200, color=root.kRed+1,
                          line_width=2, line_style=1, dphi_margin=0.0):
    """Build a TPolyLine3D (tbin, phi, radius) for a sagitta fit."""
    if fit is None: return None
    xc, yc = float(fit["xc"]), float(fit["yc"])
    R      = abs(float(fit["R"]))
    m_zr, c_zr = float(fit["m_zr"]), float(fit["c_zr"])
    if R <= 0: return None

    x_h = np.asarray(x_hits, float); y_h = np.asarray(y_hits, float)
    if len(x_h) < 2: return None

    phi_c = np.unwrap(np.arctan2(y_h - yc, x_h - xc))
    phi_start = phi_c.min() - dphi_margin
    phi_end   = phi_c.max() + dphi_margin

    pl   = root.TPolyLine3D(npoints)
    phis = []
    rads = []
    for i in range(npoints):
        t = i / float(npoints - 1)
        phi = phi_start + t*(phi_end - phi_start)
        x   = xc + R*math.cos(phi)
        y   = yc + R*math.sin(phi)
        phis.append(math.atan2(y, x))
        rads.append(math.sqrt(x*x + y*y))
    phis = np.unwrap(np.array(phis))
    for i in range(npoints):
        pl.SetPoint(i, float(m_zr*rads[i] + c_zr), float(phis[i]), float(rads[i]))
    pl.SetLineColor(color); pl.SetLineWidth(line_width); pl.SetLineStyle(line_style)
    return pl


def draw_radius_surface(hist, radius_val, color=root.kBlue, npoints=50):
    """Draw a flat mesh plane at fixed radius (module boundary indicator)."""
    tmin = hist.GetXaxis().GetXmin(); tmax = hist.GetXaxis().GetXmax()
    pmin = hist.GetYaxis().GetXmin(); pmax = hist.GetYaxis().GetXmax()
    lines = []
    for tb in np.linspace(tmin, tmax, npoints):
        pl = root.TPolyLine3D(2)
        pl.SetPoint(0, tb, pmin, radius_val); pl.SetPoint(1, tb, pmax, radius_val)
        pl.SetLineColor(color); pl.SetLineWidth(1); pl.Draw("same"); lines.append(pl)
    for ph in np.linspace(pmin, pmax, npoints):
        pl = root.TPolyLine3D(2)
        pl.SetPoint(0, tmin, ph, radius_val); pl.SetPoint(1, tmax, ph, radius_val)
        pl.SetLineColor(color); pl.SetLineWidth(1); pl.Draw("same"); lines.append(pl)
    return lines

# ── Per-object drawing helpers ────────────────────────────────────────────────

def _hits_xy(ch, gis):
    key = ch["key"]
    vx, vy = [], []
    for gi in gis:
        t, phi, r = local_coords_arr[key][gi]
        vx.append(r*math.cos(phi)); vy.append(r*math.sin(phi))
    return vx, vy

def draw_chain(hist, ch, color, line_width=2, line_style=1, dphi_margin=0.0):
    fit = ch.get("sagitta_fit")
    if fit is None or len(ch["chain_main"]) < 3: return None
    vx, vy = _hits_xy(ch, ch["chain_main"])
    pl = make_sagitta_polyline(
        f"pl_{ch['chain_id']}", fit, vx, vy,
        color=color, line_width=line_width, line_style=line_style,
        dphi_margin=dphi_margin)
    if pl: pl.Draw("same"); _draw_objs.append(pl)
    return pl

def draw_module_track(hist, mt, color, line_width=2, line_style=1, dphi_margin=0.0):
    """Draw module-track fit. Falls back to per-chain drawing if refit failed."""
    fit = mt.get("refit")
    if fit is not None:
        vx, vy = [], []
        for ch in mt["chains"]:
            x2, y2 = _hits_xy(ch, ch["chain_main"])
            vx.extend(x2); vy.extend(y2)
        pl = make_sagitta_polyline(
            f"pl_mt_{mt['track_id']}", fit, vx, vy,
            color=color, line_width=line_width, line_style=line_style,
            dphi_margin=dphi_margin)
        if pl: pl.Draw("same"); _draw_objs.append(pl)
        return pl
    # refit unavailable – draw each chain's individual fit
    for ch in mt["chains"]:
        draw_chain(hist, ch, color=color, line_width=line_width,
                   line_style=line_style, dphi_margin=dphi_margin)
    return None

def make_sagitta_polyline_radial(name, fit, r_min, r_max,
                                  npoints=300, color=root.kRed+1,
                                  line_width=2, line_style=1):
    """
    Build a TPolyLine3D spanning continuously from r_min to r_max,
    parameterised by radius rather than by angle on the fit circle.
    This draws the refit as a single unbroken curve through all modules.
    """
    if fit is None: return None
    xc, yc = float(fit["xc"]), float(fit["yc"])
    R      = abs(float(fit["R"]))
    m_zr, c_zr = float(fit["m_zr"]), float(fit["c_zr"])
    if R <= 0: return None

    phi_ref = float(fit["phi0"])   # reference direction to pick the right intersection
    radii   = np.linspace(r_min, r_max, npoints)
    pts     = []
    for r in radii:
        phi = circle_phi_at_radius(xc, yc, R, float(r), phi_ref)
        if phi is None: continue
        phi_ref = phi   # walk along the curve continuously
        pts.append((float(m_zr*r + c_zr), float(phi), float(r)))

    if len(pts) < 2: return None
    # unwrap phi to remove any 2π jumps
    raw_phis = np.unwrap([p[1] for p in pts])
    pl = root.TPolyLine3D(len(pts))
    for i, ((tbin, _, r), phi) in enumerate(zip(pts, raw_phis)):
        pl.SetPoint(i, tbin, phi, r)
    pl.SetLineColor(color); pl.SetLineWidth(line_width); pl.SetLineStyle(line_style)
    return pl


def draw_full_track(hist, ft, color, line_width=2, line_style=1, dphi_margin=0.0):
    """
    Draw full tracks with the same logic as draw_module_track.
    If the full-track refit exists, draw one sagitta polyline using all
    contributing chains' main hits. Otherwise fall back to per-chain drawing.
    """
    fit = ft.get("refit")
    if fit is not None:
        vx, vy = [], []
        for ch in ft["chains"]:
            x2, y2 = _hits_xy(ch, ch["chain_main"])
            vx.extend(x2); vy.extend(y2)
        pl = make_sagitta_polyline(
            f"pl_ft_{ft['track_id']}", fit, vx, vy,
            color=color, line_width=line_width, line_style=line_style,
            dphi_margin=dphi_margin)
        if pl: pl.Draw("same"); _draw_objs.append(pl)
        return pl
    for ch in ft["chains"]:
        draw_chain(hist, ch, color=color, line_width=line_width,
                   line_style=line_style, dphi_margin=dphi_margin)
    return None

    '''pl = make_sagitta_polyline_radial(
            f"pl_ft_{ft['track_id']}", fit,
            r_min=min(all_r), r_max=max(all_r),
            color=color, line_width=line_width, line_style=line_style)
    if pl: pl.Draw("same"); _draw_objs.append(pl)
        return pl
    # refit unavailable – draw each chain's individual fit
    for ch in ft["chains"]:
        draw_chain(hist, ch, color=color, line_width=line_width, line_style=line_style)
    return None'''
def draw_track_clusters(hist, track_cluster_dict, color=root.kBlack, marker_style=20, marker_size=1.2):
    """
    Draw ADC-weighted cluster centroids for one full track.
    Coordinates are in the same local system as the final 3D plot:
        x = tbin, y = local phi, z = radius
    """
    cls = track_cluster_dict.get("clusters", [])
    if not cls:
        return None

    pm = root.TPolyMarker3D(len(cls))
    pm.SetMarkerColor(color)
    pm.SetMarkerStyle(marker_style)
    pm.SetMarkerSize(marker_size)

    for i, cl in enumerate(sorted(cls, key=lambda x: x["radius"])):
        pm.SetPoint(i, float(cl["tbin"]), float(cl["phi"]), float(cl["radius"]))

    pm.Draw("same")
    _draw_objs.append(pm)
    return pm

def get_darker_root_color(color):
    try:
        return root.TColor.GetColorDark(color)
    except Exception:
        return color

def draw_track_clusters_dual(hist,
                             track_cluster_dict,
                             color=root.kBlue,
                             raw_marker=24,
                             corr_marker=20,
                             raw_size=1.0,
                             corr_size=1.2):
    cls = track_cluster_dict.get("clusters", [])
    if not cls:
        return None, None

    raw_pts = []
    corr_pts = []

    for cl in sorted(cls, key=lambda x: x["radius"]):
        raw_pts.append((float(cl["tbin"]), float(cl["phi"]), float(cl["radius"])))
        cc = cl.get("cluster_corrected", None)
        if cc is not None:
            corr_pts.append((float(cc["tbin"]), float(cc["phi"]), float(cc["radius"])))

    pm_raw = None
    if len(raw_pts) > 0:
        pm_raw = root.TPolyMarker3D(len(raw_pts))
        pm_raw.SetMarkerColor(color)
        pm_raw.SetMarkerStyle(raw_marker)
        pm_raw.SetMarkerSize(raw_size)
        for i, (t, p, r) in enumerate(raw_pts):
            pm_raw.SetPoint(i, t, p, r)
        pm_raw.Draw("same")
        _draw_objs.append(pm_raw)

    pm_corr = None
    dark_color = get_darker_root_color(color)
    if len(corr_pts) > 0:
        pm_corr = root.TPolyMarker3D(len(corr_pts))
        pm_corr.SetMarkerColor(dark_color)
        pm_corr.SetMarkerStyle(corr_marker)
        pm_corr.SetMarkerSize(corr_size)
        for i, (t, p, r) in enumerate(corr_pts):
            pm_corr.SetPoint(i, t, p, r)
        pm_corr.Draw("same")
        _draw_objs.append(pm_corr)

    return pm_raw, pm_corr

In [ ]:
# ── Chain display ─────────────────────────────────────────────────────────────
if Draw_chains or Draw_all_hits:
    c_chains = root.TCanvas(f"c_chains_sec{sector}_s{side}", "Chains", 1000, 1000)
    h_chains = make_hist(f"h_chains_{sector}_{side}", "Chains")
    if Draw_all_hits:
        fill_all_hits(h_chains, sector, side)
    h_chains.Draw("COLZ")
    if Draw_chains:
        for imod, chains in vchains_per_module.items():
            for idx, ch in enumerate(chains):
                color = TRACK_COLORS[idx % len(TRACK_COLORS)]
                if Draw_all_hits:
                    key = ch["key"]
                    for gi in ch["chain_consumed"]:
                        t, phi, r = local_coords_arr[key][gi]
                        h_chains.Fill(t, phi, r, int(adc_arr[key][gi]))
                draw_chain(h_chains, ch, color=color)
    c_chains.Update(); c_chains.Draw()


In [ ]:
# ── In-module track display ───────────────────────────────────────────────────
if Draw_inmodule_tracks or Draw_all_hits:
    c_inmod = root.TCanvas(f"c_inmod_sec{sector}_s{side}", "In-Module Tracks", 1000, 1000)
    h_inmod = make_hist(f"h_inmod_{sector}_{side}", "In-Module Tracks")
    if Draw_all_hits:
        fill_all_hits(h_inmod, sector, side)
    h_inmod.Draw("COLZ")
    if Draw_inmodule_tracks:
        tidx = 0
        for imod, module_tracks in tracks_per_module.items():
            for mt in module_tracks:
                color = TRACK_COLORS[tidx % len(TRACK_COLORS)]
                if Draw_all_hits:
                    for ch in mt["chains"]:
                        key = ch["key"]
                        for gi in ch["chain_consumed"]:
                            t, phi, r = local_coords_arr[key][gi]
                            h_inmod.Fill(t, phi, r, int(adc_arr[key][gi]))
                draw_module_track(h_inmod, mt, color=color)
                tidx += 1
    c_inmod.Update(); c_inmod.Draw()


In [ ]:
# ── Full track display ────────────────────────────────────────────────────────
if Draw_full_tracks or Draw_all_hits:
    c_full = root.TCanvas(f"c_full_sec{sector}_s{side}", "Full Tracks", 1000, 1000)
    h_full = make_hist(f"h_full_{sector}_{side}", "Full Tracks")
    if Draw_all_hits:
        fill_all_hits(h_full, sector, side)
    h_full.Draw("COLZ")

    if Draw_full_tracks:
        for idx, track in enumerate(full_tracks):
            color = TRACK_COLORS[idx % len(TRACK_COLORS)]

            if Draw_all_hits:
                for ch in track["chains"]:
                    key = ch["key"]
                    for gi in ch["chain_consumed"]:
                        t, phi, r = local_coords_arr[key][gi]
                        h_full.Fill(t, phi, r, int(adc_arr[key][gi]))

            draw_full_track(h_full, track, color=color)

            # draw per-layer cluster centroids on top
            '''draw_track_clusters(
                h_full,
                track_layer_clusters[idx],
                color=color,
                marker_style=20,
                marker_size=1.4
            )'''
            draw_track_clusters_dual(
                h_full,
                track_layer_clusters[idx],
                color=color,
                raw_marker=24,
                corr_marker=20,
                raw_size=1.0,
                corr_size=1.15
            )
    c_full.Update()
    c_full.Draw()

In [ ]:
# ── r·Δphi residual display from module-track refits ──────────────────────────
if Draw_chain_residuals:
    c_resid = root.TCanvas(f"c_resid_sec{sector}_s{side}", "Residuals", 1400, 700)
    c_resid.Divide(2, 1)

    h_resid = root.TH2D(
        f"h_resid_sec{sector}_s{side}",
        "r vs #Delta#phi;radius;r#Delta#phi",
        70, 20, 80,
        100, -1.0, 1.0
    )

    n_tracks_used = 0
    n_hits_filled = 0

    for imod, module_tracks in tracks_per_module.items():
        for mt in module_tracks:
            refit = mt.get("refit", None)
            if refit is None:
                continue

            # collect all hits belonging to this module track
            hit_tags = []
            layer_set = set()

            for ch in mt["chains"]:
                key = ch["key"]
                for gi in ch["chain_consumed"]:
                    gi = int(gi)
                    hit_tags.append((key, gi))
                    layer_set.add(int(local_coords_arr[key][gi][2]))

            # skip short tracks
            if len(layer_set) < 3:
                continue

            m_phir = refit.get("m_phir", None)
            c_phir = refit.get("c_phir", None)
            if m_phir is None or c_phir is None:
                continue

            n_tracks_used += 1

            for key, gi in hit_tags:
                t_h, phi_h, r_h = local_coords_arr[key][gi]
                r_h = float(r_h)
                phi_h = float(phi_h)

                phi_fit = float(m_phir) * r_h + float(c_phir)
                dphi = wrap_pi(phi_h - phi_fit)

                w = float(adc_arr[key][gi])
                h_resid.Fill(r_h, r_h * dphi, w)
                n_hits_filled += 1

    print("Residual tracks used:", n_tracks_used)
    print("Residual hits filled:", n_hits_filled)

    c_resid.cd(1)
    h_resid.Draw("colz")

    c_resid.cd(2)
    h_py = h_resid.ProjectionY(f"{h_resid.GetName()}_py")
    h_py.SetTitle("r#Delta#phi projection;r#Delta#phi;Counts")

    if h_py.GetEntries() > 5:
        sigma0 = max(float(h_py.GetRMS()), 0.03)/5
        mu0 = float(h_py.GetMean())

        fgaus = root.TF1(
            f"fgaus_{sector}_{side}",
            "gaus",
            mu0 - 5.0 * sigma0,
            mu0 + 5.0 * sigma0
        )
        fgaus.SetParameters(h_py.GetMaximum(), mu0, sigma0)
        h_py.Fit(fgaus, "RQ")
        h_py.Draw()

        print(f"Residual sigma = {fgaus.GetParameter(2):.5f} cm ± {fgaus.GetParError(2):.5f}")
    else:
        h_py.Draw()
        print("Not enough entries to fit Gaussian.")

    c_resid.Update()
    c_resid.Draw()